1+1

In [1]:
1+1

2

In [2]:
# ============================================================
# POPULATION DECODING — ARITHMETIC TASK
# CELL 1: IMPORTS, PATHS, CONSTANTS
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt

from scipy import sparse

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix


# ------------------------------------------------------------
# PROJECT PATHS
# ------------------------------------------------------------

ROOT = Path("..").resolve()

DATA_RAW = ROOT / "data" / "raw"
TABLES = ROOT / "tables"
FIGURES = ROOT / "figures"

TABLES.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)


# ------------------------------------------------------------
# SUBJECTS
# ------------------------------------------------------------

SUBJECTS = [
    "YFF",
    "YFI",
    "YFJ",
    "YFK",
    "YFL",
    "YFM",
    "YFP",
    "YFR",
    "YFS",
    "YFT",
    "YFU"
]


# ------------------------------------------------------------
# MTL REGIONS
# ------------------------------------------------------------

MTL_REGION_MAP = {
    "hpc": "HPC",
    "ent": "ENT",
    "amy": "AMY",
    "para-hpc": "PARA-HPC"
}


# ------------------------------------------------------------
# ANALYSIS PARAMETERS
# ------------------------------------------------------------

WINDOW_START_MS = 50
WINDOW_END_MS = 950

WINDOW_LENGTH_MS = (
    WINDOW_END_MS
    -
    WINDOW_START_MS
)

COMMON_TEMPORAL_BIN_MS = 60

NUMBER_CLASSES = np.arange(1, 10)

RANDOM_STATE = 0


# ------------------------------------------------------------
# DISPLAY SETTINGS
# ------------------------------------------------------------

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.width",
    140
)


# ------------------------------------------------------------
# SANITY CHECK
# ------------------------------------------------------------

print("=" * 70)
print("POPULATION DECODING NOTEBOOK INITIALIZED")
print("=" * 70)

print(f"ROOT:       {ROOT}")
print(f"DATA_RAW:   {DATA_RAW}")
print(f"TABLES:     {TABLES}")
print(f"FIGURES:    {FIGURES}")

print()

print(
    f"Subjects: {len(SUBJECTS)}"
)

print(
    f"Analysis window: "
    f"{WINDOW_START_MS}–{WINDOW_END_MS} ms "
    f"({WINDOW_LENGTH_MS} ms)"
)

print(
    f"Common temporal bin: "
    f"{COMMON_TEMPORAL_BIN_MS} ms"
)

print(
    f"Temporal bins/neuron: "
    f"{WINDOW_LENGTH_MS // COMMON_TEMPORAL_BIN_MS}"
)

print(
    f"Number classes: "
    f"{NUMBER_CLASSES.tolist()}"
)

print(
    f"Nominal 9-way chance: "
    f"{100 / len(NUMBER_CLASSES):.2f}%"
)

print()

print(
    "Raw data directory exists:",
    DATA_RAW.exists()
)

POPULATION DECODING NOTEBOOK INITIALIZED
ROOT:       C:\Users\shafi\number-simplex-reproduction
DATA_RAW:   C:\Users\shafi\number-simplex-reproduction\data\raw
TABLES:     C:\Users\shafi\number-simplex-reproduction\tables
FIGURES:    C:\Users\shafi\number-simplex-reproduction\figures

Subjects: 11
Analysis window: 50–950 ms (900 ms)
Common temporal bin: 60 ms
Temporal bins/neuron: 15
Number classes: [1, 2, 3, 4, 5, 6, 7, 8, 9]
Nominal 9-way chance: 11.11%

Raw data directory exists: True


In [3]:
# ============================================================
# CELL 2: LOAD SPIKES + REGION LABELS
# ============================================================

from scipy import sparse


# ------------------------------------------------------------
# FIND ARITHMETIC SPIKE FILE
#
# Different subjects use slightly different filenames.
# ------------------------------------------------------------

def find_arithmetic_spike_file(subject):

    arithmetic_dir = (
        DATA_RAW /
        subject /
        "arithmetic"
    )

    candidates = [
        arithmetic_dir / "spikes.mat",
        arithmetic_dir / "spikesArithmetic.mat"
    ]

    for path in candidates:

        if path.exists():
            return path

    raise FileNotFoundError(
        f"No arithmetic spike file found for {subject}"
    )


# ------------------------------------------------------------
# LOAD MATLAB v7.3 SPARSE SPIKE MATRIX
#
# Output:
#
# rows    = neurons
# columns = time samples
#
# At 1 kHz:
# one column = 1 ms
# ------------------------------------------------------------

def load_arithmetic_spikes(filepath):

    filepath = Path(filepath)

    with h5py.File(filepath, "r") as f:

        possible_variables = [
            "spikes",
            "spikesArithmetic"
        ]

        variable = None

        for candidate in possible_variables:

            if candidate in f:
                variable = candidate
                break

        if variable is None:

            raise KeyError(
                f"Could not find spikes or spikesArithmetic "
                f"in {filepath}"
            )

        g = f[variable]

        required = {
            "ir",
            "jc",
            "data"
        }

        if not required.issubset(g.keys()):

            raise ValueError(
                f"{variable} is not stored in the expected "
                f"MATLAB sparse format."
            )

        ir = (
            np.asarray(g["ir"])
            .ravel()
            .astype(np.int64)
        )

        jc = (
            np.asarray(g["jc"])
            .ravel()
            .astype(np.int64)
        )

        data = (
            np.asarray(g["data"])
            .ravel()
        )

        n_rows = int(
            np.asarray(
                g.attrs["MATLAB_sparse"]
            ).ravel()[0]
        )

        n_cols = len(jc) - 1

        spikes = sparse.csc_matrix(
            (data, ir, jc),
            shape=(n_rows, n_cols)
        )

    return spikes


# ------------------------------------------------------------
# LOAD REGION LABELS
#
# The project already has exported neuron-region CSV files.
# Search for the subject's region CSV and read it.
# ------------------------------------------------------------

def find_region_file(subject):

    subject_dir = (
        DATA_RAW /
        subject /
        "arithmetic"
    )

    candidates = [
        subject_dir / "brainArea.csv",
        subject_dir / "brainAreas.csv",
        subject_dir / "regions.csv",
        subject_dir / "region.csv"
    ]

    for path in candidates:

        if path.exists():
            return path

    # If one of the exact filenames is not present,
    # show us the available CSV files rather than guessing.
    available = list(
        subject_dir.glob("*.csv")
    )

    raise FileNotFoundError(
        f"No recognized region CSV found for {subject}.\n"
        f"Available CSV files:\n"
        +
        "\n".join(
            str(p.name)
            for p in available
        )
    )


# ------------------------------------------------------------
# TEST SPIKE LOADER ON YFF
# ------------------------------------------------------------

TEST_SUBJECT = "YFF"

spike_file = find_arithmetic_spike_file(
    TEST_SUBJECT
)

spikes_yff = load_arithmetic_spikes(
    spike_file
)


print("=" * 72)
print("YFF SPIKE LOADING CHECK")
print("=" * 72)

print(
    "Spike file:",
    spike_file.name
)

print(
    "Spike matrix shape:",
    spikes_yff.shape
)

print(
    "Number of neurons:",
    spikes_yff.shape[0]
)

print(
    "Number of time samples:",
    spikes_yff.shape[1]
)

print(
    "Nonzero entries:",
    spikes_yff.nnz
)

print(
    "Sparse format:",
    type(spikes_yff).__name__
)


# ------------------------------------------------------------
# CHECK WHICH CSV FILES EXIST
# ------------------------------------------------------------

arithmetic_dir_yff = (
    DATA_RAW /
    TEST_SUBJECT /
    "arithmetic"
)

csv_files_yff = sorted(
    arithmetic_dir_yff.glob("*.csv")
)


print("\n" + "=" * 72)
print("YFF CSV FILES")
print("=" * 72)

for path in csv_files_yff:
    print(path.name)


# ------------------------------------------------------------
# TRY TO LOCATE REGION FILE
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("REGION FILE CHECK")
print("=" * 72)

try:

    region_file_yff = find_region_file(
        TEST_SUBJECT
    )

    print(
        "Region file found:",
        region_file_yff.name
    )

except FileNotFoundError as e:

    region_file_yff = None

    print(e)

YFF SPIKE LOADING CHECK
Spike file: spikes.mat
Spike matrix shape: (54, 708276)
Number of neurons: 54
Number of time samples: 708276
Nonzero entries: 501537
Sparse format: csc_matrix

YFF CSV FILES
photoBehavEvents.csv

REGION FILE CHECK
No recognized region CSV found for YFF.
Available CSV files:
photoBehavEvents.csv


In [4]:
# ============================================================
# CELL 3: RECOVER MTL NEURONS + BUILD OPERAND PRESENTATIONS
# ============================================================

# ------------------------------------------------------------
# LOAD PREVIOUSLY VERIFIED MTL NEURON TABLE
# ------------------------------------------------------------

MTL_RESULTS_FILE = (
    TABLES /
    "arithmetic_temporal_mtl_neuron_results.csv"
)

if not MTL_RESULTS_FILE.exists():
    raise FileNotFoundError(
        f"Could not find:\n{MTL_RESULTS_FILE}"
    )

mtl_results = pd.read_csv(
    MTL_RESULTS_FILE
)

print("=" * 72)
print("MTL RESULTS TABLE")
print("=" * 72)

print(
    "Shape:",
    mtl_results.shape
)

print(
    "Columns:"
)

print(
    mtl_results.columns.tolist()
)


# ------------------------------------------------------------
# INSPECT YFF ROWS
# ------------------------------------------------------------

yff_mtl_table = (
    mtl_results[
        mtl_results["subject"] == "YFF"
    ]
    .copy()
)

print("\n" + "=" * 72)
print("YFF MTL NEURONS")
print("=" * 72)

print(
    "Number of YFF MTL rows:",
    len(yff_mtl_table)
)

print(
    yff_mtl_table.head()
)


# ------------------------------------------------------------
# DETECT NEURON INDEX COLUMN
#
# We don't guess blindly. Search common possibilities.
# ------------------------------------------------------------

possible_index_columns = [
    "neuron_idx",
    "neuron_index",
    "row_idx",
    "row_index",
    "neuron"
]

NEURON_INDEX_COLUMN = None

for col in possible_index_columns:

    if col in yff_mtl_table.columns:
        NEURON_INDEX_COLUMN = col
        break


if NEURON_INDEX_COLUMN is None:

    raise KeyError(
        "Could not identify the neuron-index column.\n"
        f"Available columns are:\n"
        f"{yff_mtl_table.columns.tolist()}"
    )


print(
    "\nNeuron index column:",
    NEURON_INDEX_COLUMN
)


# ------------------------------------------------------------
# GET YFF MTL ROW INDICES
# ------------------------------------------------------------

yff_mtl_indices = (
    yff_mtl_table[
        NEURON_INDEX_COLUMN
    ]
    .astype(int)
    .to_numpy()
)


print(
    "YFF MTL neuron count:",
    len(yff_mtl_indices)
)

print(
    "First 10 MTL raw-row indices:",
    yff_mtl_indices[:10]
)

print(
    "Minimum index:",
    yff_mtl_indices.min()
)

print(
    "Maximum index:",
    yff_mtl_indices.max()
)


# ------------------------------------------------------------
# VERIFY INDICES FIT RAW SPIKE MATRIX
# ------------------------------------------------------------

if yff_mtl_indices.min() < 0:

    raise ValueError(
        "Found negative neuron index."
    )

if yff_mtl_indices.max() >= spikes_yff.shape[0]:

    raise ValueError(
        "Neuron indices exceed the raw spike-matrix rows.\n"
        "This may indicate MATLAB 1-based indices rather than "
        "Python 0-based indices."
    )


# ------------------------------------------------------------
# OPERAND ONSET FUNCTION
#
# Most subjects:
#
# operationFirst = True
#     Cue1 = operator
#     Cue2 = operand1
#     Cue3 = operand2
#
# operationFirst = False
#     Cue1 = operand1
#     Cue2 = operand2
#     Cue3 = operator
#
# YFR / YFS:
#     Cue1 = operand1
#     Cue2 = operator
#     Cue3 = operand2
# ------------------------------------------------------------

def add_operand_onsets(df, subject):

    df = df.copy()

    if subject in ["YFR", "YFS"]:

        df["operand1_onset"] = df["tCue1"]
        df["operand2_onset"] = df["tCue3"]

    else:

        op_first = (
            df["operationFirst"]
            .astype(bool)
        )

        df["operand1_onset"] = np.where(
            op_first,
            df["tCue2"],
            df["tCue1"]
        )

        df["operand2_onset"] = np.where(
            op_first,
            df["tCue3"],
            df["tCue2"]
        )

    return df


# ------------------------------------------------------------
# BUILD POOLED OPERAND PRESENTATIONS
# ------------------------------------------------------------

def prepare_arithmetic_subject(
    subject,
    spikes
):

    behavior_path = (
        DATA_RAW /
        subject /
        "arithmetic" /
        "photoBehavEvents.csv"
    )

    if not behavior_path.exists():

        raise FileNotFoundError(
            behavior_path
        )

    behav = pd.read_csv(
        behavior_path
    )

    behav = add_operand_onsets(
        behav,
        subject
    )

    pooled_rows = []

    for trial_idx, row in behav.iterrows():

        # Operand 1
        if 1 <= row["cue1"] <= 9:

            pooled_rows.append(
                {
                    "trial_idx": trial_idx,
                    "trial": row["trial"],
                    "operand_position": 1,
                    "number": int(row["cue1"]),
                    "onset": row["operand1_onset"]
                }
            )

        # Operand 2
        if 1 <= row["cue2"] <= 9:

            pooled_rows.append(
                {
                    "trial_idx": trial_idx,
                    "trial": row["trial"],
                    "operand_position": 2,
                    "number": int(row["cue2"]),
                    "onset": row["operand2_onset"]
                }
            )


    pooled = pd.DataFrame(
        pooled_rows
    )


    pooled["onset_sample"] = (
        pooled["onset"]
        .round()
        .astype(int)
    )


    pooled["start_sample"] = (
        pooled["onset_sample"]
        +
        WINDOW_START_MS
    )


    pooled["end_sample"] = (
        pooled["onset_sample"]
        +
        WINDOW_END_MS
    )


    pooled["valid_window"] = (
        (pooled["start_sample"] >= 0)
        &
        (
            pooled["end_sample"]
            <=
            spikes.shape[1]
        )
    )


    return behav, pooled


# ------------------------------------------------------------
# TEST ON YFF
# ------------------------------------------------------------

behav_yff, pooled_yff = (
    prepare_arithmetic_subject(
        "YFF",
        spikes_yff
    )
)


print("\n" + "=" * 72)
print("YFF BEHAVIOR / PRESENTATION CHECK")
print("=" * 72)

print(
    "Original behavioral trials:",
    len(behav_yff)
)

print(
    "Pooled operand presentations:",
    len(pooled_yff)
)

print(
    "Valid 900-ms windows:",
    pooled_yff["valid_window"].sum()
)

print()

print(
    "Operand-position counts:"
)

print(
    pooled_yff[
        "operand_position"
    ].value_counts().sort_index()
)

print()

print(
    "Number-class counts:"
)

print(
    pooled_yff[
        "number"
    ].value_counts().sort_index()
)

print()

print(
    "First 8 pooled presentations:"
)

print(
    pooled_yff.head(8).to_string(
        index=False
    )
)

MTL RESULTS TABLE
Shape: (554, 14)
Columns:
['subject', 'neuron', 'region_raw', 'region', 'n_trials', 'n_presentations', 'best_bin_ms', 'best_gamma', 'temporal_accuracy', 'permutation_p', 'n_perm_equal_or_better', 'coding', 'valid', 'invalid_reason']

YFF MTL NEURONS
Number of YFF MTL rows: 37
  subject  neuron region_raw region  n_trials  n_presentations  best_bin_ms  best_gamma  temporal_accuracy  permutation_p  \
0     YFF       8        ent    ENT       100              109         60.0         0.5           0.146789       0.288557   
1     YFF       9        hpc    HPC       100              109         90.0         0.2           0.183486       0.044776   
2     YFF      10        hpc    HPC       100              109         75.0         0.5           0.128440       0.318408   
3     YFF      11        hpc    HPC       100              109        450.0         0.2           0.155963       0.124378   
4     YFF      12        hpc    HPC       100              109        900.0     

In [5]:
# ============================================================
# CHECK HOW THE "neuron" COLUMN IS STORED
# ============================================================

print(
    yff_mtl_table[
        [
            "subject",
            "neuron",
            "region_raw",
            "region"
        ]
    ].head(15).to_string(index=False)
)

print("\nData type of neuron column:")
print(
    yff_mtl_table["neuron"].dtype
)

print("\nFirst 15 neuron values:")
print(
    yff_mtl_table["neuron"].head(15).tolist()
)

subject  neuron region_raw region
    YFF       8        ent    ENT
    YFF       9        hpc    HPC
    YFF      10        hpc    HPC
    YFF      11        hpc    HPC
    YFF      12        hpc    HPC
    YFF      13        hpc    HPC
    YFF      14        hpc    HPC
    YFF      15        hpc    HPC
    YFF      16        hpc    HPC
    YFF      26        hpc    HPC
    YFF      27        hpc    HPC
    YFF      28        hpc    HPC
    YFF      29        hpc    HPC
    YFF      30        hpc    HPC
    YFF      31        hpc    HPC

Data type of neuron column:
int64

First 15 neuron values:
[8, 9, 10, 11, 12, 13, 14, 15, 16, 26, 27, 28, 29, 30, 31]


In [6]:
# ============================================================
# CELL 4: BUILD POPULATION FR + TEMPORAL FEATURE MATRICES
# ============================================================

def make_population_features(
    spikes,
    pooled_table,
    neuron_indices,
    temporal_bin_ms=60
):
    """
    Build two population representations using the SAME
    operand presentations and SAME neurons.

    Returns
    -------
    X_fr : array, shape (n_presentations, n_neurons)
        Total spike count in the 900-ms window for each neuron.

    X_temporal : array, shape
        (n_presentations, n_neurons * n_bins)
        Temporal spike counts, concatenated neuron by neuron.

    y : array
        Number labels 1..9.

    metadata : DataFrame
        Information about each presentation.
    """

    window_length = (
        WINDOW_END_MS
        -
        WINDOW_START_MS
    )

    if window_length % temporal_bin_ms != 0:
        raise ValueError(
            f"{window_length} ms window is not divisible "
            f"by {temporal_bin_ms} ms."
        )

    n_bins = (
        window_length //
        temporal_bin_ms
    )

    neuron_indices = np.asarray(
        neuron_indices,
        dtype=int
    )

    X_fr = []
    X_temporal = []
    y = []
    metadata_rows = []


    for _, row in pooled_table.iterrows():

        if not row["valid_window"]:
            continue

        start = int(
            row["start_sample"]
        )

        end = int(
            row["end_sample"]
        )


        # ----------------------------------------------------
        # Population response:
        #
        # rows    = neurons
        # columns = milliseconds in the 900-ms window
        #
        # shape = (N neurons, 900)
        # ----------------------------------------------------

        response = (
            spikes[
                neuron_indices,
                start:end
            ]
            .toarray()
            .astype(float)
        )


        if response.shape != (
            len(neuron_indices),
            window_length
        ):

            continue


        # ----------------------------------------------------
        # FIRING-RATE / TOTAL-SPIKE-COUNT REPRESENTATION
        #
        # One number per neuron.
        #
        # shape = (N neurons,)
        # ----------------------------------------------------

        fr_vector = (
            response.sum(axis=1)
        )


        # ----------------------------------------------------
        # TEMPORAL REPRESENTATION
        #
        # Example for 60 ms:
        #
        # 900 / 60 = 15 bins per neuron
        #
        # reshape:
        # (N neurons, 900)
        #
        # ->
        #
        # (N neurons, 15 bins, 60 ms)
        #
        # then sum within each temporal bin.
        # ----------------------------------------------------

        temporal_binned = (
            response
            .reshape(
                len(neuron_indices),
                n_bins,
                temporal_bin_ms
            )
            .sum(axis=2)
        )


        # ----------------------------------------------------
        # CONCATENATE NEURONS
        #
        # neuron 1 bins,
        # then neuron 2 bins,
        # ...
        #
        # shape = N * n_bins
        # ----------------------------------------------------

        temporal_vector = (
            temporal_binned.reshape(-1)
        )


        X_fr.append(
            fr_vector
        )

        X_temporal.append(
            temporal_vector
        )

        y.append(
            int(row["number"])
        )

        metadata_rows.append(
            {
                "trial_idx":
                    int(row["trial_idx"]),

                "trial":
                    row["trial"],

                "operand_position":
                    int(row["operand_position"]),

                "number":
                    int(row["number"]),

                "onset_sample":
                    int(row["onset_sample"])
            }
        )


    X_fr = np.asarray(
        X_fr,
        dtype=float
    )

    X_temporal = np.asarray(
        X_temporal,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=int
    )

    metadata = pd.DataFrame(
        metadata_rows
    )


    return (
        X_fr,
        X_temporal,
        y,
        metadata
    )


# ------------------------------------------------------------
# BUILD YFF MATRICES
# ------------------------------------------------------------

X_fr_yff, X_temp_yff, y_yff, meta_yff = (
    make_population_features(
        spikes=spikes_yff,
        pooled_table=pooled_yff,
        neuron_indices=yff_mtl_indices,
        temporal_bin_ms=COMMON_TEMPORAL_BIN_MS
    )
)


# ------------------------------------------------------------
# SANITY CHECK
# ------------------------------------------------------------

print("=" * 72)
print("YFF POPULATION FEATURE MATRICES")
print("=" * 72)

print(
    "Number of MTL neurons:",
    len(yff_mtl_indices)
)

print(
    "Number of presentations:",
    len(y_yff)
)

print()

print(
    "FR matrix shape:",
    X_fr_yff.shape
)

print(
    "Temporal matrix shape:",
    X_temp_yff.shape
)

print()

print(
    "Expected FR features:",
    len(yff_mtl_indices)
)

print(
    "Expected temporal features:",
    len(yff_mtl_indices)
    *
    (
        WINDOW_LENGTH_MS //
        COMMON_TEMPORAL_BIN_MS
    )
)

print()

print(
    "Contains NaN in FR matrix:",
    np.isnan(X_fr_yff).any()
)

print(
    "Contains NaN in temporal matrix:",
    np.isnan(X_temp_yff).any()
)

print()

print(
    "Class counts:"
)

print(
    pd.Series(y_yff)
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# INSPECT FIRST PRESENTATION
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("FIRST PRESENTATION")
print("=" * 72)

print(
    meta_yff.iloc[0]
)

print()

print(
    "FR population vector shape:",
    X_fr_yff[0].shape
)

print(
    "FR vector:"
)

print(
    X_fr_yff[0]
)

print()

print(
    "Temporal population vector shape:",
    X_temp_yff[0].shape
)

print(
    "First 45 temporal features "
    "(first 3 neurons × 15 bins):"
)

print(
    X_temp_yff[0][:45]
)

YFF POPULATION FEATURE MATRICES
Number of MTL neurons: 37
Number of presentations: 109

FR matrix shape: (109, 37)
Temporal matrix shape: (109, 555)

Expected FR features: 37
Expected temporal features: 555

Contains NaN in FR matrix: False
Contains NaN in temporal matrix: False

Class counts:
1    10
2    15
3    11
4     7
5     7
6    16
7    17
8    12
9    14
Name: count, dtype: int64

FIRST PRESENTATION
trial_idx              0
trial                  1
operand_position       1
number                 5
onset_sample        6089
Name: 0, dtype: int64

FR population vector shape: (37,)
FR vector:
[ 1. 16. 12. 20. 20. 15. 24. 12. 16.  8.  0. 13.  0.  5.  0. 10. 13. 10.
  9.  0. 18. 10.  3. 41.  0. 26.  0. 19.  0.  8.  0. 31.  0. 16.  0. 21.
  0.]

Temporal population vector shape: (555,)
First 45 temporal features (first 3 neurons × 15 bins):
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 2. 2. 1. 0. 4.
 0. 1. 0. 3. 1. 1. 1. 0. 0. 1. 1. 1. 2. 0. 1. 0. 0. 0. 3. 1. 1.]


In [7]:
# ============================================================
# CELL 5: MULTINOMIAL POPULATION DECODER
# SAME CV SPLITS FOR FR AND TEMPORAL REPRESENTATIONS
# ============================================================

def make_stratified_cv_splits(
    y,
    random_state=0,
    max_splits=10
):
    """
    Create stratified CV splits.

    The number of folds is reduced if the smallest number
    class has fewer than 10 observations.
    """

    y = np.asarray(y)

    class_counts = (
        pd.Series(y)
        .value_counts()
        .sort_index()
    )

    min_class_count = int(
        class_counts.min()
    )

    n_splits = min(
        max_splits,
        min_class_count
    )

    if n_splits < 2:
        raise ValueError(
            "Not enough observations per class "
            "for cross-validation."
        )

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    splits = list(
        cv.split(
            np.zeros(len(y)),
            y
        )
    )

    return splits


def population_logistic_cv(
    X,
    y,
    splits
):
    """
    Multinomial population decoding using L2-regularized
    logistic regression.

    StandardScaler is fitted ONLY on training data
    within each fold.

    Parameters
    ----------
    X : array
        samples x features

    y : array
        labels 1..9

    splits : list
        Precomputed train/test indices.

    Returns
    -------
    accuracy : float
        Overall held-out accuracy.

    y_pred : array
        Prediction for every held-out observation.

    fold_table : DataFrame
        Fold-level results.
    """

    X = np.asarray(
        X,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=int
    )


    # --------------------------------------------------------
    # BASIC CHECKS
    # --------------------------------------------------------

    if np.isnan(X).any():
        raise ValueError(
            "X contains NaN values."
        )

    if np.isinf(X).any():
        raise ValueError(
            "X contains infinite values."
        )


    y_pred = np.full(
        len(y),
        -1,
        dtype=int
    )

    fold_rows = []


    # --------------------------------------------------------
    # CROSS-VALIDATION
    # --------------------------------------------------------

    for fold_number, (
        train_idx,
        test_idx
    ) in enumerate(
        splits,
        start=1
    ):

        X_train = X[train_idx]
        X_test = X[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]


        # ----------------------------------------------------
        # Pipeline:
        #
        # 1. Standardize features using TRAINING DATA ONLY
        #
        # 2. Fit L2-regularized multinomial logistic regression
        # ----------------------------------------------------

        model = Pipeline(
            [
                (
                    "scaler",
                    StandardScaler()
                ),

                (
                    "classifier",
                    LogisticRegression(
                        penalty="l2",
                        C=1.0,
                        solver="lbfgs",
                        max_iter=5000
                    )
                )
            ]
        )


        model.fit(
            X_train,
            y_train
        )


        fold_pred = model.predict(
            X_test
        )


        y_pred[test_idx] = fold_pred


        fold_accuracy = np.mean(
            fold_pred == y_test
        )


        fold_rows.append(
            {
                "fold": fold_number,
                "n_train": len(train_idx),
                "n_test": len(test_idx),
                "accuracy": fold_accuracy
            }
        )


    # --------------------------------------------------------
    # POOLED HELD-OUT ACCURACY
    # --------------------------------------------------------

    if np.any(y_pred == -1):
        raise RuntimeError(
            "Some observations never received "
            "a held-out prediction."
        )


    accuracy = np.mean(
        y_pred == y
    )


    fold_table = pd.DataFrame(
        fold_rows
    )


    return (
        accuracy,
        y_pred,
        fold_table
    )


# ============================================================
# CREATE ONE COMMON SET OF YFF CV SPLITS
# ============================================================

splits_yff = make_stratified_cv_splits(
    y_yff,
    random_state=RANDOM_STATE
)


print("=" * 72)
print("YFF CROSS-VALIDATION")
print("=" * 72)

print(
    "Number of CV folds:",
    len(splits_yff)
)

print()

for fold, (
    train_idx,
    test_idx
) in enumerate(
    splits_yff,
    start=1
):

    print(
        f"Fold {fold:2d}: "
        f"train = {len(train_idx):3d}, "
        f"test = {len(test_idx):3d}"
    )


# ============================================================
# FIRING-RATE POPULATION DECODER
# ============================================================

fr_accuracy_yff, fr_pred_yff, fr_folds_yff = (
    population_logistic_cv(
        X=X_fr_yff,
        y=y_yff,
        splits=splits_yff
    )
)


# ============================================================
# TEMPORAL POPULATION DECODER
# ============================================================

temp_accuracy_yff, temp_pred_yff, temp_folds_yff = (
    population_logistic_cv(
        X=X_temp_yff,
        y=y_yff,
        splits=splits_yff
    )
)


# ============================================================
# RESULTS
# ============================================================

chance_accuracy = (
    1 /
    len(NUMBER_CLASSES)
)


print("\n" + "=" * 72)
print("YFF POPULATION DECODING RESULTS")
print("=" * 72)

print(
    f"Chance level:              "
    f"{chance_accuracy * 100:.2f}%"
)

print(
    f"Firing-rate accuracy:      "
    f"{fr_accuracy_yff * 100:.2f}%"
)

print(
    f"Temporal accuracy (60 ms): "
    f"{temp_accuracy_yff * 100:.2f}%"
)

print(
    f"Temporal - FR:             "
    f"{(temp_accuracy_yff - fr_accuracy_yff) * 100:+.2f} "
    f"percentage points"
)

print()

print(
    "Correct FR predictions:",
    np.sum(fr_pred_yff == y_yff),
    "/",
    len(y_yff)
)

print(
    "Correct temporal predictions:",
    np.sum(temp_pred_yff == y_yff),
    "/",
    len(y_yff)
)

YFF CROSS-VALIDATION
Number of CV folds: 7

Fold  1: train =  93, test =  16
Fold  2: train =  93, test =  16
Fold  3: train =  93, test =  16
Fold  4: train =  93, test =  16
Fold  5: train =  94, test =  15
Fold  6: train =  94, test =  15
Fold  7: train =  94, test =  15


c:\Users\shafi\number-simplex-reproduction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\shafi\number-simplex-reproduction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\sha


YFF POPULATION DECODING RESULTS
Chance level:              11.11%
Firing-rate accuracy:      10.09%
Temporal accuracy (60 ms): 15.60%
Temporal - FR:             +5.50 percentage points

Correct FR predictions: 11 / 109
Correct temporal predictions: 17 / 109


c:\Users\shafi\number-simplex-reproduction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\shafi\number-simplex-reproduction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\sha

In [8]:
# ============================================================
# CELL 6: YFF PERMUTATION TEST
#
# Tests:
#   1. Is FR decoding above its shuffled null?
#   2. Is temporal decoding above its shuffled null?
#   3. Is Temporal - FR larger than expected under shuffled labels?
# ============================================================


# ------------------------------------------------------------
# REDEFINE DECODER WITHOUT sklearn DEPRECATION WARNING
# ------------------------------------------------------------

def population_logistic_cv(
    X,
    y,
    splits
):

    X = np.asarray(
        X,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=int
    )

    if np.isnan(X).any():
        raise ValueError(
            "X contains NaN values."
        )

    if np.isinf(X).any():
        raise ValueError(
            "X contains infinite values."
        )


    y_pred = np.full(
        len(y),
        -1,
        dtype=int
    )


    for train_idx, test_idx in splits:

        X_train = X[train_idx]
        X_test = X[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]


        model = Pipeline(
            [
                (
                    "scaler",
                    StandardScaler()
                ),

                (
                    "classifier",
                    LogisticRegression(
                        C=1.0,
                        solver="lbfgs",
                        max_iter=5000
                    )
                )
            ]
        )


        model.fit(
            X_train,
            y_train
        )

        y_pred[test_idx] = (
            model.predict(
                X_test
            )
        )


    accuracy = np.mean(
        y_pred == y
    )


    return accuracy, y_pred


# ------------------------------------------------------------
# OBSERVED RESULTS
# ------------------------------------------------------------

fr_accuracy_obs, fr_pred_obs = (
    population_logistic_cv(
        X_fr_yff,
        y_yff,
        splits_yff
    )
)

temp_accuracy_obs, temp_pred_obs = (
    population_logistic_cv(
        X_temp_yff,
        y_yff,
        splits_yff
    )
)

delta_obs = (
    temp_accuracy_obs
    -
    fr_accuracy_obs
)


# ------------------------------------------------------------
# PERMUTATION TEST
# ------------------------------------------------------------

N_PERMUTATIONS = 200

rng = np.random.default_rng(
    RANDOM_STATE
)


null_fr = np.zeros(
    N_PERMUTATIONS
)

null_temp = np.zeros(
    N_PERMUTATIONS
)

null_delta = np.zeros(
    N_PERMUTATIONS
)


print("=" * 72)
print("RUNNING YFF PERMUTATION TEST")
print("=" * 72)


for perm in range(
    N_PERMUTATIONS
):

    # --------------------------------------------------------
    # Shuffle number labels.
    #
    # Neural activity X stays exactly the same.
    # --------------------------------------------------------

    y_shuffled = rng.permutation(
        y_yff
    )


    # --------------------------------------------------------
    # FR NULL DECODING
    # --------------------------------------------------------

    fr_perm_accuracy, _ = (
        population_logistic_cv(
            X_fr_yff,
            y_shuffled,
            splits_yff
        )
    )


    # --------------------------------------------------------
    # TEMPORAL NULL DECODING
    # --------------------------------------------------------

    temp_perm_accuracy, _ = (
        population_logistic_cv(
            X_temp_yff,
            y_shuffled,
            splits_yff
        )
    )


    null_fr[perm] = (
        fr_perm_accuracy
    )

    null_temp[perm] = (
        temp_perm_accuracy
    )

    null_delta[perm] = (
        temp_perm_accuracy
        -
        fr_perm_accuracy
    )


    if (
        (perm + 1) % 20 == 0
    ):

        print(
            f"Completed "
            f"{perm + 1:3d} / "
            f"{N_PERMUTATIONS}"
        )


# ------------------------------------------------------------
# EXACT MONTE-CARLO STYLE P VALUES
# ------------------------------------------------------------

p_fr = (
    1
    +
    np.sum(
        null_fr
        >=
        fr_accuracy_obs
    )
) / (
    N_PERMUTATIONS
    +
    1
)


p_temp = (
    1
    +
    np.sum(
        null_temp
        >=
        temp_accuracy_obs
    )
) / (
    N_PERMUTATIONS
    +
    1
)


p_delta = (
    1
    +
    np.sum(
        null_delta
        >=
        delta_obs
    )
) / (
    N_PERMUTATIONS
    +
    1
)


# ------------------------------------------------------------
# PRINT RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("YFF PERMUTATION RESULTS")
print("=" * 72)

print(
    f"Observed FR accuracy:          "
    f"{100 * fr_accuracy_obs:.2f}%"
)

print(
    f"Mean shuffled FR accuracy:     "
    f"{100 * null_fr.mean():.2f}%"
)

print(
    f"FR permutation p:              "
    f"{p_fr:.4f}"
)

print()

print(
    f"Observed temporal accuracy:    "
    f"{100 * temp_accuracy_obs:.2f}%"
)

print(
    f"Mean shuffled temporal acc.:   "
    f"{100 * null_temp.mean():.2f}%"
)

print(
    f"Temporal permutation p:        "
    f"{p_temp:.4f}"
)

print()

print(
    f"Observed Temporal - FR:        "
    f"{100 * delta_obs:+.2f} pp"
)

print(
    f"Mean shuffled Temporal - FR:   "
    f"{100 * null_delta.mean():+.2f} pp"
)

print(
    f"Temporal-vs-FR permutation p:  "
    f"{p_delta:.4f}"
)

RUNNING YFF PERMUTATION TEST
Completed  20 / 200
Completed  40 / 200
Completed  60 / 200
Completed  80 / 200
Completed 100 / 200
Completed 120 / 200
Completed 140 / 200
Completed 160 / 200
Completed 180 / 200
Completed 200 / 200

YFF PERMUTATION RESULTS
Observed FR accuracy:          10.09%
Mean shuffled FR accuracy:     11.68%
FR permutation p:              0.7264

Observed temporal accuracy:    15.60%
Mean shuffled temporal acc.:   11.29%
Temporal permutation p:        0.1194

Observed Temporal - FR:        +5.50 pp
Mean shuffled Temporal - FR:   -0.39 pp
Temporal-vs-FR permutation p:  0.1244


-----

All 11 subjects 

In [9]:
# ============================================================
# CELL 7: POPULATION DECODING FOR ALL 11 SUBJECTS
# ============================================================


def get_subject_mtl_indices(
    subject,
    mtl_results
):
    """
    Recover raw spike-matrix row indices for one subject
    from the previously verified MTL neuron table.
    """

    subject_table = (
        mtl_results[
            mtl_results["subject"] == subject
        ]
        .copy()
    )

    if len(subject_table) == 0:
        raise ValueError(
            f"No MTL neurons found for {subject}"
        )

    indices = (
        subject_table["neuron"]
        .astype(int)
        .to_numpy()
    )

    return indices, subject_table


# ------------------------------------------------------------
# RUN ALL SUBJECTS
# ------------------------------------------------------------

all_subject_rows = []


print("=" * 85)
print("ALL-SUBJECT POPULATION DECODING")
print("=" * 85)


for subject in SUBJECTS:

    print(
        f"\nProcessing {subject} ..."
    )


    # --------------------------------------------------------
    # LOAD SPIKES
    # --------------------------------------------------------

    spike_file = (
        find_arithmetic_spike_file(
            subject
        )
    )

    spikes = (
        load_arithmetic_spikes(
            spike_file
        )
    )


    # --------------------------------------------------------
    # MTL NEURONS
    # --------------------------------------------------------

    mtl_indices, subject_mtl_table = (
        get_subject_mtl_indices(
            subject,
            mtl_results
        )
    )


    # --------------------------------------------------------
    # SAFETY CHECK
    # --------------------------------------------------------

    if mtl_indices.max() >= spikes.shape[0]:

        raise ValueError(
            f"{subject}: MTL neuron index exceeds "
            f"spike matrix rows."
        )


    # --------------------------------------------------------
    # BEHAVIOR / OPERAND PRESENTATIONS
    # --------------------------------------------------------

    behav, pooled = (
        prepare_arithmetic_subject(
            subject,
            spikes
        )
    )


    # --------------------------------------------------------
    # BUILD BOTH POPULATION REPRESENTATIONS
    # --------------------------------------------------------

    X_fr, X_temp, y, metadata = (
        make_population_features(
            spikes=spikes,
            pooled_table=pooled,
            neuron_indices=mtl_indices,
            temporal_bin_ms=COMMON_TEMPORAL_BIN_MS
        )
    )


    # --------------------------------------------------------
    # SAME CV SPLITS FOR BOTH REPRESENTATIONS
    # --------------------------------------------------------

    splits = (
        make_stratified_cv_splits(
            y,
            random_state=RANDOM_STATE
        )
    )


    # --------------------------------------------------------
    # FR DECODING
    # --------------------------------------------------------

    fr_accuracy, fr_pred = (
        population_logistic_cv(
            X_fr,
            y,
            splits
        )
    )


    # --------------------------------------------------------
    # TEMPORAL DECODING
    # --------------------------------------------------------

    temp_accuracy, temp_pred = (
        population_logistic_cv(
            X_temp,
            y,
            splits
        )
    )


    delta = (
        temp_accuracy
        -
        fr_accuracy
    )


    # --------------------------------------------------------
    # STORE RESULT
    # --------------------------------------------------------

    all_subject_rows.append(
        {
            "subject":
                subject,

            "n_mtl_neurons":
                len(mtl_indices),

            "n_presentations":
                len(y),

            "n_fr_features":
                X_fr.shape[1],

            "n_temporal_features":
                X_temp.shape[1],

            "n_cv_folds":
                len(splits),

            "fr_accuracy":
                fr_accuracy,

            "temporal_accuracy":
                temp_accuracy,

            "temporal_minus_fr":
                delta,

            "fr_correct":
                int(
                    np.sum(
                        fr_pred == y
                    )
                ),

            "temporal_correct":
                int(
                    np.sum(
                        temp_pred == y
                    )
                )
        }
    )


    print(
        f"  neurons       = {len(mtl_indices)}"
    )

    print(
        f"  presentations = {len(y)}"
    )

    print(
        f"  FR            = "
        f"{100 * fr_accuracy:.2f}%"
    )

    print(
        f"  Temporal      = "
        f"{100 * temp_accuracy:.2f}%"
    )

    print(
        f"  Temp - FR     = "
        f"{100 * delta:+.2f} pp"
    )


# ------------------------------------------------------------
# CREATE SUMMARY TABLE
# ------------------------------------------------------------

population_results = pd.DataFrame(
    all_subject_rows
)


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

output_file = (
    TABLES /
    "arithmetic_population_fr_vs_temporal_60ms.csv"
)

population_results.to_csv(
    output_file,
    index=False
)


# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

display_table = (
    population_results[
        [
            "subject",
            "n_mtl_neurons",
            "n_presentations",
            "fr_accuracy",
            "temporal_accuracy",
            "temporal_minus_fr"
        ]
    ]
    .copy()
)

display_table[
    "fr_accuracy"
] *= 100

display_table[
    "temporal_accuracy"
] *= 100

display_table[
    "temporal_minus_fr"
] *= 100


print("\n" + "=" * 85)
print("ALL-SUBJECT SUMMARY")
print("=" * 85)

print(
    display_table.to_string(
        index=False,
        formatters={
            "fr_accuracy":
                "{:.2f}%".format,

            "temporal_accuracy":
                "{:.2f}%".format,

            "temporal_minus_fr":
                "{:+.2f} pp".format
        }
    )
)


print("\n" + "=" * 85)
print("GROUP SUMMARY")
print("=" * 85)

print(
    f"Mean FR accuracy:       "
    f"{100 * population_results['fr_accuracy'].mean():.2f}%"
)

print(
    f"Mean temporal accuracy: "
    f"{100 * population_results['temporal_accuracy'].mean():.2f}%"
)

print(
    f"Mean temporal - FR:     "
    f"{100 * population_results['temporal_minus_fr'].mean():+.2f} pp"
)

print(
    f"Subjects Temp > FR:     "
    f"{(population_results['temporal_minus_fr'] > 0).sum()}"
    f"/{len(population_results)}"
)

print()

print(
    "Saved to:"
)

print(
    output_file
)

ALL-SUBJECT POPULATION DECODING

Processing YFF ...
  neurons       = 37
  presentations = 109
  FR            = 10.09%
  Temporal      = 15.60%
  Temp - FR     = +5.50 pp

Processing YFI ...
  neurons       = 29
  presentations = 104
  FR            = 11.54%
  Temporal      = 20.19%
  Temp - FR     = +8.65 pp

Processing YFJ ...
  neurons       = 45
  presentations = 114
  FR            = 16.67%
  Temporal      = 16.67%
  Temp - FR     = +0.00 pp

Processing YFK ...
  neurons       = 44
  presentations = 114
  FR            = 14.91%
  Temporal      = 18.42%
  Temp - FR     = +3.51 pp

Processing YFL ...
  neurons       = 57
  presentations = 114
  FR            = 10.53%
  Temporal      = 12.28%
  Temp - FR     = +1.75 pp

Processing YFM ...
  neurons       = 61
  presentations = 114
  FR            = 14.91%
  Temporal      = 20.18%
  Temp - FR     = +5.26 pp

Processing YFP ...
  neurons       = 43
  presentations = 158
  FR            = 13.92%
  Temporal      = 12.66%
  Temp - FR    

In [10]:
# ============================================================
# CELL 8: GROUP-LEVEL TEMPORAL VS FR COMPARISON
# ============================================================

from scipy.stats import ttest_rel, wilcoxon


# ------------------------------------------------------------
# SUBJECT-LEVEL DIFFERENCES
# ------------------------------------------------------------

subject_delta = (
    population_results["temporal_accuracy"]
    -
    population_results["fr_accuracy"]
)


# ------------------------------------------------------------
# PAIRED T TEST
# ------------------------------------------------------------

t_stat, t_p = ttest_rel(
    population_results["temporal_accuracy"],
    population_results["fr_accuracy"]
)


# ------------------------------------------------------------
# WILCOXON SIGNED-RANK TEST
#
# Nonparametric sensitivity check.
# ------------------------------------------------------------

w_stat, w_p = wilcoxon(
    population_results["temporal_accuracy"],
    population_results["fr_accuracy"],
    zero_method="wilcox"
)


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

mean_fr = (
    population_results["fr_accuracy"].mean()
)

mean_temp = (
    population_results["temporal_accuracy"].mean()
)

mean_delta = (
    subject_delta.mean()
)

sem_delta = (
    subject_delta.std(ddof=1)
    /
    np.sqrt(len(subject_delta))
)


print("=" * 72)
print("GROUP-LEVEL POPULATION DECODING COMPARISON")
print("=" * 72)

print(
    f"Subjects:                   "
    f"{len(population_results)}"
)

print()

print(
    f"Mean FR accuracy:           "
    f"{100 * mean_fr:.2f}%"
)

print(
    f"Mean temporal accuracy:     "
    f"{100 * mean_temp:.2f}%"
)

print(
    f"Mean Temporal - FR:         "
    f"{100 * mean_delta:+.2f} pp"
)

print(
    f"SEM of difference:          "
    f"{100 * sem_delta:.2f} pp"
)

print()

print(
    f"Temporal > FR:              "
    f"{(subject_delta > 0).sum()}/11"
)

print(
    f"Temporal = FR:              "
    f"{(subject_delta == 0).sum()}/11"
)

print(
    f"Temporal < FR:              "
    f"{(subject_delta < 0).sum()}/11"
)

print()

print(
    f"Paired t-test:              "
    f"t(10) = {t_stat:.4f}, "
    f"p = {t_p:.6f}"
)

print(
    f"Wilcoxon signed-rank:       "
    f"W = {w_stat:.4f}, "
    f"p = {w_p:.6f}"
)


# ------------------------------------------------------------
# SHOW ALL SUBJECTS INCLUDING YFU
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("SUBJECT DIFFERENCES")
print("=" * 72)

for _, row in population_results.iterrows():

    print(
        f"{row['subject']}: "
        f"FR = {100 * row['fr_accuracy']:6.2f}%   "
        f"Temporal = {100 * row['temporal_accuracy']:6.2f}%   "
        f"Delta = {100 * row['temporal_minus_fr']:+6.2f} pp"
    )

GROUP-LEVEL POPULATION DECODING COMPARISON
Subjects:                   11

Mean FR accuracy:           12.59%
Mean temporal accuracy:     14.87%
Mean Temporal - FR:         +2.28 pp
SEM of difference:          0.99 pp

Temporal > FR:              7/11
Temporal = FR:              1/11
Temporal < FR:              3/11

Paired t-test:              t(10) = 2.3081, p = 0.043647
Wilcoxon signed-rank:       W = 9.0000, p = 0.064453

SUBJECT DIFFERENCES
YFF: FR =  10.09%   Temporal =  15.60%   Delta =  +5.50 pp
YFI: FR =  11.54%   Temporal =  20.19%   Delta =  +8.65 pp
YFJ: FR =  16.67%   Temporal =  16.67%   Delta =  +0.00 pp
YFK: FR =  14.91%   Temporal =  18.42%   Delta =  +3.51 pp
YFL: FR =  10.53%   Temporal =  12.28%   Delta =  +1.75 pp
YFM: FR =  14.91%   Temporal =  20.18%   Delta =  +5.26 pp
YFP: FR =  13.92%   Temporal =  12.66%   Delta =  -1.27 pp
YFR: FR =  12.78%   Temporal =  15.04%   Delta =  +2.26 pp
YFS: FR =   9.85%   Temporal =   7.58%   Delta =  -2.27 pp
YFT: FR =  11.95%  

---

UsE LDA components instead of 60ms bin size 

I performed subject-wise 9-class population decoding using all simultaneously recorded MTL neurons. Using identical cross-validation splits and L2-regularized multinomial logistic regression, mean decoding accuracy was 12.59% using total 900-ms spike counts and 14.87% using 60-ms temporal bins, compared with nominal 9-way chance of 11.11%. The mean within-subject temporal advantage was 2.28 percentage points (7/11 subjects positive; paired \(t(10)=2.31,p=0.044\); Wilcoxon \(p=0.064\)). Thus the temporal advantage is suggestive but not yet robust, and permutation controls / alternative temporal representations are needed.

In [11]:
# ============================================================
# CELL 9: POPULATION DECODING USING PER-NEURON
#         TEMPORAL LDA COMPONENTS
#
# IMPORTANT:
# LDA is fitted ONLY on the training data inside each CV fold.
#
# Current version:
#   temporal bin = 60 ms for every neuron
#   shrinkage gamma = 0.5
#   retain up to 3 LDA components / neuron
# ============================================================

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis


def make_single_neuron_temporal_matrix(
    spikes,
    pooled_table,
    neuron_idx,
    temporal_bin_ms=60
):
    """
    Create temporal-bin features for ONE neuron.

    Output
    ------
    X : shape (n_presentations, n_bins)
    y : number labels
    """

    window_length = (
        WINDOW_END_MS
        -
        WINDOW_START_MS
    )

    if window_length % temporal_bin_ms != 0:
        raise ValueError(
            "Temporal bin does not divide the analysis window."
        )

    n_bins = (
        window_length //
        temporal_bin_ms
    )

    X = []
    y = []


    for _, row in pooled_table.iterrows():

        if not row["valid_window"]:
            continue

        start = int(
            row["start_sample"]
        )

        end = int(
            row["end_sample"]
        )

        response = (
            spikes[
                neuron_idx,
                start:end
            ]
            .toarray()
            .ravel()
            .astype(float)
        )

        if len(response) != window_length:
            continue


        binned = (
            response
            .reshape(
                n_bins,
                temporal_bin_ms
            )
            .sum(axis=1)
        )

        X.append(
            binned
        )

        y.append(
            int(row["number"])
        )


    return (
        np.asarray(X, dtype=float),
        np.asarray(y, dtype=int)
    )


def lda_component_population_cv(
    spikes,
    pooled_table,
    neuron_indices,
    y,
    splits,
    temporal_bin_ms=60,
    gamma=0.5,
    max_components=3
):
    """
    Outer CV population decoding.

    For each outer fold:

        1. For each neuron:
           - construct its temporal-bin features
           - fit LDA using TRAINING presentations only
           - project train and held-out data
           - retain up to max_components

        2. Concatenate the LDA components across neurons

        3. Fit multinomial logistic regression on population
           LDA representation

        4. Predict held-out number

    Returns
    -------
    accuracy
    y_pred
    fold_table
    """

    neuron_indices = np.asarray(
        neuron_indices,
        dtype=int
    )

    y = np.asarray(
        y,
        dtype=int
    )


    # --------------------------------------------------------
    # PRECOMPUTE RAW TEMPORAL FEATURES FOR EACH NEURON
    #
    # This step uses NO labels for fitting.
    # It simply extracts spike counts.
    # --------------------------------------------------------

    neuron_feature_matrices = []

    for neuron_idx in neuron_indices:

        X_neuron, y_check = (
            make_single_neuron_temporal_matrix(
                spikes=spikes,
                pooled_table=pooled_table,
                neuron_idx=neuron_idx,
                temporal_bin_ms=temporal_bin_ms
            )
        )

        if not np.array_equal(
            y_check,
            y
        ):
            raise ValueError(
                "Presentation alignment mismatch."
            )

        neuron_feature_matrices.append(
            X_neuron
        )


    # --------------------------------------------------------
    # OUTER HELD-OUT PREDICTIONS
    # --------------------------------------------------------

    y_pred = np.full(
        len(y),
        -1,
        dtype=int
    )

    fold_rows = []


    for fold_number, (
        train_idx,
        test_idx
    ) in enumerate(
        splits,
        start=1
    ):

        train_component_blocks = []
        test_component_blocks = []

        neurons_used = 0
        neurons_skipped = 0


        # ====================================================
        # EACH NEURON GETS ITS OWN TEMPORAL LDA
        # ====================================================

        for X_neuron in neuron_feature_matrices:

            X_train_neuron = (
                X_neuron[train_idx]
            )

            X_test_neuron = (
                X_neuron[test_idx]
            )

            y_train = (
                y[train_idx]
            )


            # ------------------------------------------------
            # If a neuron has absolutely no temporal variation
            # in the training fold, it contains nothing for
            # LDA to learn.
            # ------------------------------------------------

            if np.all(
                np.var(
                    X_train_neuron,
                    axis=0
                )
                ==
                0
            ):

                neurons_skipped += 1
                continue


            # ------------------------------------------------
            # Number of possible discriminant dimensions:
            #
            # <= classes - 1
            # <= temporal feature dimensions
            # <= requested max_components
            # ------------------------------------------------

            n_classes_train = (
                len(
                    np.unique(
                        y_train
                    )
                )
            )

            n_components = min(
                max_components,
                n_classes_train - 1,
                X_train_neuron.shape[1]
            )


            if n_components < 1:

                neurons_skipped += 1
                continue


            # ------------------------------------------------
            # SHRINKAGE LDA
            #
            # sklearn's eigen solver permits:
            #   shrinkage = float in [0,1]
            #
            # gamma = 0.5 here.
            # ------------------------------------------------

            lda = LinearDiscriminantAnalysis(
                solver="eigen",
                shrinkage=gamma,
                n_components=n_components
            )


            try:

                lda.fit(
                    X_train_neuron,
                    y_train
                )

                train_components = (
                    lda.transform(
                        X_train_neuron
                    )
                )

                test_components = (
                    lda.transform(
                        X_test_neuron
                    )
                )


            except Exception:

                neurons_skipped += 1
                continue


            # ------------------------------------------------
            # SAFETY
            # ------------------------------------------------

            if (
                np.isnan(train_components).any()
                or
                np.isnan(test_components).any()
            ):

                neurons_skipped += 1
                continue


            train_component_blocks.append(
                train_components
            )

            test_component_blocks.append(
                test_components
            )

            neurons_used += 1


        # ====================================================
        # CONCATENATE ALL NEURON LDA COMPONENTS
        # ====================================================

        if len(train_component_blocks) == 0:

            raise RuntimeError(
                f"Fold {fold_number}: "
                f"no neurons produced usable LDA components."
            )


        X_train_population = np.hstack(
            train_component_blocks
        )

        X_test_population = np.hstack(
            test_component_blocks
        )


        # ====================================================
        # POPULATION MULTINOMIAL LOGISTIC REGRESSION
        # ====================================================

        population_model = Pipeline(
            [
                (
                    "scaler",
                    StandardScaler()
                ),

                (
                    "classifier",
                    LogisticRegression(
                        C=1.0,
                        solver="lbfgs",
                        max_iter=5000
                    )
                )
            ]
        )


        population_model.fit(
            X_train_population,
            y[train_idx]
        )


        fold_pred = (
            population_model.predict(
                X_test_population
            )
        )


        y_pred[test_idx] = (
            fold_pred
        )


        fold_accuracy = np.mean(
            fold_pred
            ==
            y[test_idx]
        )


        fold_rows.append(
            {
                "fold":
                    fold_number,

                "n_train":
                    len(train_idx),

                "n_test":
                    len(test_idx),

                "n_neurons_used":
                    neurons_used,

                "n_neurons_skipped":
                    neurons_skipped,

                "n_population_features":
                    X_train_population.shape[1],

                "accuracy":
                    fold_accuracy
            }
        )


    if np.any(
        y_pred == -1
    ):

        raise RuntimeError(
            "Some held-out samples were never predicted."
        )


    overall_accuracy = np.mean(
        y_pred
        ==
        y
    )


    return (
        overall_accuracy,
        y_pred,
        pd.DataFrame(
            fold_rows
        )
    )


# ============================================================
# RUN YFF
# ============================================================

lda_accuracy_yff, lda_pred_yff, lda_folds_yff = (
    lda_component_population_cv(
        spikes=spikes_yff,
        pooled_table=pooled_yff,
        neuron_indices=yff_mtl_indices,
        y=y_yff,
        splits=splits_yff,
        temporal_bin_ms=60,
        gamma=0.5,
        max_components=3
    )
)


# ============================================================
# RESULTS
# ============================================================

print("=" * 75)
print("YFF — THREE POPULATION REPRESENTATIONS")
print("=" * 75)

print(
    f"Chance:                     "
    f"{100 / 9:.2f}%"
)

print()

print(
    f"FR population:               "
    f"{100 * fr_accuracy_yff:.2f}%"
)

print(
    f"Raw temporal population:     "
    f"{100 * temp_accuracy_yff:.2f}%"
)

print(
    f"LDA-component population:    "
    f"{100 * lda_accuracy_yff:.2f}%"
)

print()

print(
    f"LDA - FR:                    "
    f"{100 * (lda_accuracy_yff - fr_accuracy_yff):+.2f} pp"
)

print(
    f"LDA - raw temporal:          "
    f"{100 * (lda_accuracy_yff - temp_accuracy_yff):+.2f} pp"
)


print("\n" + "=" * 75)
print("FOLD DETAILS")
print("=" * 75)

print(
    lda_folds_yff.to_string(
        index=False
    )
)

YFF — THREE POPULATION REPRESENTATIONS
Chance:                     11.11%

FR population:               10.09%
Raw temporal population:     15.60%
LDA-component population:    20.18%

LDA - FR:                    +10.09 pp
LDA - raw temporal:          +4.59 pp

FOLD DETAILS
 fold  n_train  n_test  n_neurons_used  n_neurons_skipped  n_population_features  accuracy
    1       93      16              37                  0                    111  0.062500
    2       93      16              37                  0                    111  0.250000
    3       93      16              37                  0                    111  0.062500
    4       93      16              37                  0                    111  0.187500
    5       94      15              37                  0                    111  0.200000
    6       94      15              37                  0                    111  0.266667
    7       94      15              37                  0                    111  0.4000

The procedure for every outer CV fold is:

$$ \boxed{ \begin{array}{c} \text{Training presentations}\\ \downarrow\\ \text{fit temporal LDA separately for each neuron}\\ \downarrow\\ \text{project training presentations}\\ \text{project held-out presentations}\\ \downarrow\\ \text{concatenate neuron components}\\ \downarrow\\ \text{population multinomial logistic regression}\\ \downarrow\\ \text{predict held-out number} \end{array}} $$



For all sujects

In [12]:
# ============================================================
# CELL 10: LDA-COMPONENT POPULATION DECODING
#          FOR ALL 11 SUBJECTS
# ============================================================

all_lda_rows = []


print("=" * 85)
print("ALL-SUBJECT LDA-COMPONENT POPULATION DECODING")
print("=" * 85)


for subject in SUBJECTS:

    print(
        f"\nProcessing {subject} ..."
    )


    # --------------------------------------------------------
    # LOAD SPIKES
    # --------------------------------------------------------

    spike_file = (
        find_arithmetic_spike_file(
            subject
        )
    )

    spikes = (
        load_arithmetic_spikes(
            spike_file
        )
    )


    # --------------------------------------------------------
    # GET MTL NEURONS
    # --------------------------------------------------------

    mtl_indices, subject_mtl_table = (
        get_subject_mtl_indices(
            subject,
            mtl_results
        )
    )


    # --------------------------------------------------------
    # PREPARE BEHAVIOR
    # --------------------------------------------------------

    behav, pooled = (
        prepare_arithmetic_subject(
            subject,
            spikes
        )
    )


    # --------------------------------------------------------
    # BUILD RAW POPULATION FEATURES
    #
    # We mainly need y here, but rebuilding this guarantees
    # identical presentation ordering to the previous analysis.
    # --------------------------------------------------------

    X_fr, X_temp, y, metadata = (
        make_population_features(
            spikes=spikes,
            pooled_table=pooled,
            neuron_indices=mtl_indices,
            temporal_bin_ms=COMMON_TEMPORAL_BIN_MS
        )
    )


    # --------------------------------------------------------
    # SAME CV SPLITS AS PREVIOUS ANALYSIS
    # --------------------------------------------------------

    splits = (
        make_stratified_cv_splits(
            y,
            random_state=RANDOM_STATE
        )
    )


    # --------------------------------------------------------
    # LDA-COMPONENT POPULATION DECODING
    # --------------------------------------------------------

    lda_accuracy, lda_pred, lda_folds = (
        lda_component_population_cv(
            spikes=spikes,
            pooled_table=pooled,
            neuron_indices=mtl_indices,
            y=y,
            splits=splits,
            temporal_bin_ms=COMMON_TEMPORAL_BIN_MS,
            gamma=0.5,
            max_components=3
        )
    )


    # --------------------------------------------------------
    # GET PREVIOUS FR + RAW TEMPORAL RESULTS
    # --------------------------------------------------------

    previous_row = (
        population_results[
            population_results["subject"]
            ==
            subject
        ]
        .iloc[0]
    )


    fr_accuracy = (
        previous_row[
            "fr_accuracy"
        ]
    )

    raw_temp_accuracy = (
        previous_row[
            "temporal_accuracy"
        ]
    )


    # --------------------------------------------------------
    # FOLD DIAGNOSTICS
    # --------------------------------------------------------

    mean_neurons_used = (
        lda_folds[
            "n_neurons_used"
        ].mean()
    )

    mean_neurons_skipped = (
        lda_folds[
            "n_neurons_skipped"
        ].mean()
    )

    mean_lda_features = (
        lda_folds[
            "n_population_features"
        ].mean()
    )


    # --------------------------------------------------------
    # STORE
    # --------------------------------------------------------

    all_lda_rows.append(
        {
            "subject":
                subject,

            "n_mtl_neurons":
                len(mtl_indices),

            "n_presentations":
                len(y),

            "fr_accuracy":
                fr_accuracy,

            "raw_temporal_accuracy":
                raw_temp_accuracy,

            "lda_component_accuracy":
                lda_accuracy,

            "raw_temporal_minus_fr":
                raw_temp_accuracy
                -
                fr_accuracy,

            "lda_minus_fr":
                lda_accuracy
                -
                fr_accuracy,

            "lda_minus_raw_temporal":
                lda_accuracy
                -
                raw_temp_accuracy,

            "mean_neurons_used":
                mean_neurons_used,

            "mean_neurons_skipped":
                mean_neurons_skipped,

            "mean_lda_features":
                mean_lda_features
        }
    )


    print(
        f"  FR          = "
        f"{100 * fr_accuracy:.2f}%"
    )

    print(
        f"  Raw temporal= "
        f"{100 * raw_temp_accuracy:.2f}%"
    )

    print(
        f"  LDA comp.   = "
        f"{100 * lda_accuracy:.2f}%"
    )

    print(
        f"  LDA - FR    = "
        f"{100 * (lda_accuracy - fr_accuracy):+.2f} pp"
    )

    print(
        f"  neurons used/skipped ≈ "
        f"{mean_neurons_used:.1f} / "
        f"{mean_neurons_skipped:.1f}"
    )


# ============================================================
# CREATE TABLE
# ============================================================

population_three_methods = (
    pd.DataFrame(
        all_lda_rows
    )
)


# ============================================================
# SAVE
# ============================================================

output_file_3methods = (
    TABLES /
    "arithmetic_population_three_representations.csv"
)

population_three_methods.to_csv(
    output_file_3methods,
    index=False
)


# ============================================================
# DISPLAY SUBJECT RESULTS
# ============================================================

print("\n" + "=" * 85)
print("THREE-REPRESENTATION SUBJECT SUMMARY")
print("=" * 85)


for _, row in (
    population_three_methods.iterrows()
):

    print(
        f"{row['subject']}:  "
        f"FR = {100 * row['fr_accuracy']:6.2f}%   "
        f"RawTemp = {100 * row['raw_temporal_accuracy']:6.2f}%   "
        f"LDA = {100 * row['lda_component_accuracy']:6.2f}%   "
        f"LDA-FR = {100 * row['lda_minus_fr']:+6.2f} pp"
    )


# ============================================================
# GROUP SUMMARY
# ============================================================

print("\n" + "=" * 85)
print("GROUP SUMMARY")
print("=" * 85)


mean_fr = (
    population_three_methods[
        "fr_accuracy"
    ].mean()
)

mean_raw = (
    population_three_methods[
        "raw_temporal_accuracy"
    ].mean()
)

mean_lda = (
    population_three_methods[
        "lda_component_accuracy"
    ].mean()
)


print(
    f"Mean FR accuracy:             "
    f"{100 * mean_fr:.2f}%"
)

print(
    f"Mean raw temporal accuracy:   "
    f"{100 * mean_raw:.2f}%"
)

print(
    f"Mean LDA-component accuracy:  "
    f"{100 * mean_lda:.2f}%"
)

print()

print(
    f"Mean RawTemp - FR:            "
    f"{100 * (mean_raw - mean_fr):+.2f} pp"
)

print(
    f"Mean LDA - FR:                "
    f"{100 * (mean_lda - mean_fr):+.2f} pp"
)

print(
    f"Mean LDA - RawTemp:           "
    f"{100 * (mean_lda - mean_raw):+.2f} pp"
)

print()

print(
    "Subjects LDA > FR:           ",
    int(
        (
            population_three_methods[
                "lda_minus_fr"
            ]
            >
            0
        ).sum()
    ),
    "/11"
)

print(
    "Subjects LDA > RawTemp:      ",
    int(
        (
            population_three_methods[
                "lda_minus_raw_temporal"
            ]
            >
            0
        ).sum()
    ),
    "/11"
)

print()

print(
    "Saved to:"
)

print(
    output_file_3methods
)

ALL-SUBJECT LDA-COMPONENT POPULATION DECODING

Processing YFF ...
  FR          = 10.09%
  Raw temporal= 15.60%
  LDA comp.   = 20.18%
  LDA - FR    = +10.09 pp
  neurons used/skipped ≈ 37.0 / 0.0

Processing YFI ...
  FR          = 11.54%
  Raw temporal= 20.19%
  LDA comp.   = 18.27%
  LDA - FR    = +6.73 pp
  neurons used/skipped ≈ 29.0 / 0.0

Processing YFJ ...
  FR          = 16.67%
  Raw temporal= 16.67%
  LDA comp.   = 10.53%
  LDA - FR    = -6.14 pp
  neurons used/skipped ≈ 45.0 / 0.0

Processing YFK ...
  FR          = 14.91%
  Raw temporal= 18.42%
  LDA comp.   = 21.05%
  LDA - FR    = +6.14 pp
  neurons used/skipped ≈ 44.0 / 0.0

Processing YFL ...
  FR          = 10.53%
  Raw temporal= 12.28%
  LDA comp.   = 6.14%
  LDA - FR    = -4.39 pp
  neurons used/skipped ≈ 56.0 / 1.0

Processing YFM ...
  FR          = 14.91%
  Raw temporal= 20.18%
  LDA comp.   = 13.16%
  LDA - FR    = -1.75 pp
  neurons used/skipped ≈ 60.5 / 0.5

Processing YFP ...
  FR          = 13.92%
  Raw tempo

do permutation test 

FR vs 60 ms temporal

In [13]:
# ============================================================
# CELL 10: ALL-SUBJECT PERMUTATION TEST
#
# Tests for each subject:
#
#   1. FR decoding
#   2. 60-ms temporal decoding
#   3. Temporal - FR advantage
#
# Also constructs a group-level permutation null for the
# mean Temporal - FR difference across the 11 subjects.
# ============================================================

N_POP_PERMUTATIONS = 100

permutation_rows = []

# Store each subject's null delta distribution so that
# we can construct a group-level null afterward.
all_subject_null_deltas = []

observed_group_deltas = []


print("=" * 88)
print("ALL-SUBJECT POPULATION PERMUTATION TEST")
print("=" * 88)

print(
    f"Permutations per subject: "
    f"{N_POP_PERMUTATIONS}"
)


# ============================================================
# LOOP OVER SUBJECTS
# ============================================================

for subject_number, subject in enumerate(
    SUBJECTS,
    start=1
):

    print("\n" + "-" * 88)

    print(
        f"[{subject_number:2d}/11] "
        f"Processing {subject}"
    )

    print("-" * 88)


    # --------------------------------------------------------
    # LOAD SPIKES
    # --------------------------------------------------------

    spike_file = (
        find_arithmetic_spike_file(
            subject
        )
    )

    spikes = (
        load_arithmetic_spikes(
            spike_file
        )
    )


    # --------------------------------------------------------
    # GET MTL NEURONS
    # --------------------------------------------------------

    mtl_indices, subject_mtl_table = (
        get_subject_mtl_indices(
            subject,
            mtl_results
        )
    )


    # --------------------------------------------------------
    # PREPARE PRESENTATIONS
    # --------------------------------------------------------

    behav, pooled = (
        prepare_arithmetic_subject(
            subject,
            spikes
        )
    )


    # --------------------------------------------------------
    # BUILD FR + TEMPORAL FEATURES
    #
    # SAME presentations and SAME neurons.
    # --------------------------------------------------------

    X_fr, X_temp, y, metadata = (
        make_population_features(
            spikes=spikes,
            pooled_table=pooled,
            neuron_indices=mtl_indices,
            temporal_bin_ms=COMMON_TEMPORAL_BIN_MS
        )
    )


    # ========================================================
    # OBSERVED DECODING
    # ========================================================

    observed_splits = (
        make_stratified_cv_splits(
            y,
            random_state=RANDOM_STATE
        )
    )


    fr_obs, _ = (
        population_logistic_cv(
            X_fr,
            y,
            observed_splits
        )
    )


    temp_obs, _ = (
        population_logistic_cv(
            X_temp,
            y,
            observed_splits
        )
    )


    delta_obs = (
        temp_obs
        -
        fr_obs
    )


    observed_group_deltas.append(
        delta_obs
    )


    # ========================================================
    # NULL DISTRIBUTIONS
    # ========================================================

    null_fr = np.zeros(
        N_POP_PERMUTATIONS
    )

    null_temp = np.zeros(
        N_POP_PERMUTATIONS
    )

    null_delta = np.zeros(
        N_POP_PERMUTATIONS
    )


    # Different reproducible RNG stream for each subject
    rng = np.random.default_rng(
        RANDOM_STATE
        +
        subject_number * 1000
    )


    for perm in range(
        N_POP_PERMUTATIONS
    ):

        # ----------------------------------------------------
        # Shuffle labels
        # ----------------------------------------------------

        y_perm = (
            rng.permutation(
                y
            )
        )


        # ----------------------------------------------------
        # Rebuild stratified folds for shuffled labels.
        #
        # FR and temporal use EXACTLY the same folds for
        # this particular permutation.
        # ----------------------------------------------------

        perm_splits = (
            make_stratified_cv_splits(
                y_perm,
                random_state=RANDOM_STATE + perm
            )
        )


        # ----------------------------------------------------
        # FR NULL
        # ----------------------------------------------------

        fr_perm, _ = (
            population_logistic_cv(
                X_fr,
                y_perm,
                perm_splits
            )
        )


        # ----------------------------------------------------
        # TEMPORAL NULL
        # ----------------------------------------------------

        temp_perm, _ = (
            population_logistic_cv(
                X_temp,
                y_perm,
                perm_splits
            )
        )


        null_fr[perm] = (
            fr_perm
        )

        null_temp[perm] = (
            temp_perm
        )

        null_delta[perm] = (
            temp_perm
            -
            fr_perm
        )


        if (
            (perm + 1) % 20 == 0
        ):

            print(
                f"    permutations: "
                f"{perm + 1:3d}/"
                f"{N_POP_PERMUTATIONS}"
            )


    # ========================================================
    # PERMUTATION P VALUES
    # ========================================================

    p_fr = (
        1
        +
        np.sum(
            null_fr
            >=
            fr_obs
        )
    ) / (
        N_POP_PERMUTATIONS
        +
        1
    )


    p_temp = (
        1
        +
        np.sum(
            null_temp
            >=
            temp_obs
        )
    ) / (
        N_POP_PERMUTATIONS
        +
        1
    )


    p_delta = (
        1
        +
        np.sum(
            null_delta
            >=
            delta_obs
        )
    ) / (
        N_POP_PERMUTATIONS
        +
        1
    )


    # --------------------------------------------------------
    # STORE NULL DELTA FOR GROUP TEST
    # --------------------------------------------------------

    all_subject_null_deltas.append(
        null_delta.copy()
    )


    # --------------------------------------------------------
    # STORE SUMMARY
    # --------------------------------------------------------

    permutation_rows.append(
        {
            "subject":
                subject,

            "n_mtl_neurons":
                len(mtl_indices),

            "n_presentations":
                len(y),

            "fr_accuracy":
                fr_obs,

            "fr_null_mean":
                null_fr.mean(),

            "fr_perm_p":
                p_fr,

            "temporal_accuracy":
                temp_obs,

            "temporal_null_mean":
                null_temp.mean(),

            "temporal_perm_p":
                p_temp,

            "temporal_minus_fr":
                delta_obs,

            "delta_null_mean":
                null_delta.mean(),

            "delta_perm_p":
                p_delta
        }
    )


    # --------------------------------------------------------
    # SUBJECT RESULT
    # --------------------------------------------------------

    print()

    print(
        f"    FR:       "
        f"{100 * fr_obs:6.2f}%   "
        f"null={100 * null_fr.mean():6.2f}%   "
        f"p={p_fr:.4f}"
    )

    print(
        f"    Temporal: "
        f"{100 * temp_obs:6.2f}%   "
        f"null={100 * null_temp.mean():6.2f}%   "
        f"p={p_temp:.4f}"
    )

    print(
        f"    Delta:    "
        f"{100 * delta_obs:+6.2f} pp   "
        f"null={100 * null_delta.mean():+6.2f} pp   "
        f"p={p_delta:.4f}"
    )


# ============================================================
# SUBJECT-LEVEL SUMMARY TABLE
# ============================================================

population_perm_results = (
    pd.DataFrame(
        permutation_rows
    )
)


# ============================================================
# GROUP-LEVEL PERMUTATION TEST FOR TEMPORAL - FR
#
# Each column corresponds to one permutation iteration.
# We average the independently shuffled subject deltas.
# ============================================================

null_delta_matrix = np.vstack(
    all_subject_null_deltas
)

group_null_delta = (
    null_delta_matrix.mean(
        axis=0
    )
)

observed_group_delta = (
    np.mean(
        observed_group_deltas
    )
)


group_delta_p = (
    1
    +
    np.sum(
        group_null_delta
        >=
        observed_group_delta
    )
) / (
    N_POP_PERMUTATIONS
    +
    1
)


# ============================================================
# SAVE RESULTS
# ============================================================

summary_file = (
    TABLES /
    "arithmetic_population_fr_vs_temporal_60ms_permutation.csv"
)

population_perm_results.to_csv(
    summary_file,
    index=False
)


group_null_file = (
    TABLES /
    "arithmetic_population_temporal_minus_fr_group_null.csv"
)

pd.DataFrame(
    {
        "permutation":
            np.arange(
                1,
                N_POP_PERMUTATIONS + 1
            ),

        "mean_null_temporal_minus_fr":
            group_null_delta
    }
).to_csv(
    group_null_file,
    index=False
)


# ============================================================
# FINAL DISPLAY
# ============================================================

print("\n" + "=" * 100)
print("SUBJECT-WISE PERMUTATION SUMMARY")
print("=" * 100)


for _, row in population_perm_results.iterrows():

    print(
        f"{row['subject']}:  "
        f"FR={100 * row['fr_accuracy']:6.2f}% "
        f"(p={row['fr_perm_p']:.4f})   "
        f"Temp={100 * row['temporal_accuracy']:6.2f}% "
        f"(p={row['temporal_perm_p']:.4f})   "
        f"Delta={100 * row['temporal_minus_fr']:+6.2f} pp "
        f"(p={row['delta_perm_p']:.4f})"
    )


print("\n" + "=" * 100)
print("GROUP-LEVEL TEMPORAL ADVANTAGE")
print("=" * 100)

print(
    f"Observed mean Temporal - FR:  "
    f"{100 * observed_group_delta:+.2f} pp"
)

print(
    f"Mean permutation null delta:  "
    f"{100 * group_null_delta.mean():+.2f} pp"
)

print(
    f"Group permutation p-value:    "
    f"{group_delta_p:.4f}"
)

print()

print(
    "Subjects with significant FR decoding "
    "(p < 0.05):",
    int(
        (
            population_perm_results[
                "fr_perm_p"
            ]
            < 0.05
        ).sum()
    ),
    "/ 11"
)

print(
    "Subjects with significant temporal decoding "
    "(p < 0.05):",
    int(
        (
            population_perm_results[
                "temporal_perm_p"
            ]
            < 0.05
        ).sum()
    ),
    "/ 11"
)

print(
    "Subjects with significant Temporal > FR "
    "(p < 0.05):",
    int(
        (
            population_perm_results[
                "delta_perm_p"
            ]
            < 0.05
        ).sum()
    ),
    "/ 11"
)


print("\nSaved:")
print(summary_file)
print(group_null_file)

ALL-SUBJECT POPULATION PERMUTATION TEST
Permutations per subject: 100

----------------------------------------------------------------------------------------
[ 1/11] Processing YFF
----------------------------------------------------------------------------------------
    permutations:  20/100
    permutations:  40/100
    permutations:  60/100
    permutations:  80/100
    permutations: 100/100

    FR:        10.09%   null= 12.40%   p=0.7921
    Temporal:  15.60%   null= 13.18%   p=0.2574
    Delta:     +5.50 pp   null= +0.78 pp   p=0.1485

----------------------------------------------------------------------------------------
[ 2/11] Processing YFI
----------------------------------------------------------------------------------------
    permutations:  20/100
    permutations:  40/100
    permutations:  60/100
    permutations:  80/100
    permutations: 100/100

    FR:        11.54%   null= 12.41%   p=0.6535
    Temporal:  20.19%   null= 15.33%   p=0.1188
    Delta:     +8.65

In [14]:
# ============================================================
# PRINT FINAL POPULATION PERMUTATION SUMMARY
# ============================================================

print("=" * 100)
print("SUBJECT-WISE PERMUTATION SUMMARY")
print("=" * 100)

for _, row in population_perm_results.iterrows():

    print(
        f"{row['subject']}:  "
        f"FR={100 * row['fr_accuracy']:6.2f}% "
        f"(null={100 * row['fr_null_mean']:6.2f}%, "
        f"p={row['fr_perm_p']:.4f})   "
        f"Temp={100 * row['temporal_accuracy']:6.2f}% "
        f"(null={100 * row['temporal_null_mean']:6.2f}%, "
        f"p={row['temporal_perm_p']:.4f})   "
        f"Delta={100 * row['temporal_minus_fr']:+6.2f} pp "
        f"(null={100 * row['delta_null_mean']:+6.2f} pp, "
        f"p={row['delta_perm_p']:.4f})"
    )


print("\n" + "=" * 100)
print("GROUP-LEVEL TEMPORAL ADVANTAGE")
print("=" * 100)

print(
    f"Mean FR accuracy:             "
    f"{100 * population_perm_results['fr_accuracy'].mean():.2f}%"
)

print(
    f"Mean temporal accuracy:       "
    f"{100 * population_perm_results['temporal_accuracy'].mean():.2f}%"
)

print(
    f"Observed mean Temp - FR:      "
    f"{100 * observed_group_delta:+.2f} pp"
)

print(
    f"Mean permutation null delta:  "
    f"{100 * group_null_delta.mean():+.2f} pp"
)

print(
    f"Group permutation p-value:    "
    f"{group_delta_p:.4f}"
)


print("\n" + "=" * 100)
print("NUMBER OF SIGNIFICANT SUBJECTS")
print("=" * 100)

print(
    "FR decoding p < 0.05:          ",
    (population_perm_results["fr_perm_p"] < 0.05).sum(),
    "/ 11"
)

print(
    "Temporal decoding p < 0.05:    ",
    (population_perm_results["temporal_perm_p"] < 0.05).sum(),
    "/ 11"
)

print(
    "Temporal > FR delta p < 0.05:  ",
    (population_perm_results["delta_perm_p"] < 0.05).sum(),
    "/ 11"
)

SUBJECT-WISE PERMUTATION SUMMARY
YFF:  FR= 10.09% (null= 12.40%, p=0.7921)   Temp= 15.60% (null= 13.18%, p=0.2574)   Delta= +5.50 pp (null= +0.78 pp, p=0.1485)
YFI:  FR= 11.54% (null= 12.41%, p=0.6535)   Temp= 20.19% (null= 15.33%, p=0.1188)   Delta= +8.65 pp (null= +2.91 pp, p=0.1485)
YFJ:  FR= 16.67% (null= 13.53%, p=0.2376)   Temp= 16.67% (null= 13.92%, p=0.2277)   Delta= +0.00 pp (null= +0.39 pp, p=0.5941)
YFK:  FR= 14.91% (null= 12.97%, p=0.3168)   Temp= 18.42% (null= 13.96%, p=0.1881)   Delta= +3.51 pp (null= +0.99 pp, p=0.3366)
YFL:  FR= 10.53% (null= 12.94%, p=0.8218)   Temp= 12.28% (null= 13.66%, p=0.6832)   Delta= +1.75 pp (null= +0.72 pp, p=0.5050)
YFM:  FR= 14.91% (null= 12.53%, p=0.2772)   Temp= 20.18% (null= 14.20%, p=0.0594)   Delta= +5.26 pp (null= +1.68 pp, p=0.2178)
YFP:  FR= 13.92% (null= 12.25%, p=0.2970)   Temp= 12.66% (null= 13.42%, p=0.6139)   Delta= -1.27 pp (null= +1.17 pp, p=0.6931)
YFR:  FR= 12.78% (null= 11.32%, p=0.2970)   Temp= 15.04% (null= 11.86%, p=0.08

we do not have statistically significant population decoding evidence from this analysis.

In [19]:
# ============================================================
# FIXED LDA COMPONENT POPULATION CV
#
# Key correction:
# During permutation testing, y is intentionally shuffled.
# Therefore we CANNOT require y_check == y.
#
# We only require the extracted feature rows to have the
# same number of presentations.
# ============================================================

def lda_component_population_cv(
    spikes,
    pooled_table,
    neuron_indices,
    y,
    splits,
    temporal_bin_ms=60,
    gamma=0.5,
    max_components=3
):

    neuron_indices = np.asarray(
        neuron_indices,
        dtype=int
    )

    y = np.asarray(
        y,
        dtype=int
    )


    # --------------------------------------------------------
    # PRECOMPUTE RAW TEMPORAL FEATURES FOR EACH NEURON
    # --------------------------------------------------------

    neuron_feature_matrices = []

    for neuron_idx in neuron_indices:

        X_neuron, y_original = (
            make_single_neuron_temporal_matrix(
                spikes=spikes,
                pooled_table=pooled_table,
                neuron_idx=neuron_idx,
                temporal_bin_ms=temporal_bin_ms
            )
        )


        # ----------------------------------------------------
        # IMPORTANT:
        #
        # y_original contains the TRUE behavioral labels.
        #
        # During permutation testing, y contains SHUFFLED
        # labels, so we must NOT compare them element-by-element.
        #
        # We only verify that rows/presentations align in length.
        # ----------------------------------------------------

        if len(X_neuron) != len(y):

            raise ValueError(
                f"Presentation count mismatch: "
                f"X has {len(X_neuron)} rows, "
                f"y has {len(y)} labels."
            )


        neuron_feature_matrices.append(
            X_neuron
        )


    # --------------------------------------------------------
    # HELD-OUT PREDICTIONS
    # --------------------------------------------------------

    y_pred = np.full(
        len(y),
        -1,
        dtype=int
    )

    fold_rows = []


    # ========================================================
    # OUTER CROSS-VALIDATION
    # ========================================================

    for fold_number, (
        train_idx,
        test_idx
    ) in enumerate(
        splits,
        start=1
    ):

        train_component_blocks = []
        test_component_blocks = []

        neurons_used = 0
        neurons_skipped = 0


        # ====================================================
        # FIT ONE TEMPORAL LDA PER NEURON
        # ====================================================

        for X_neuron in neuron_feature_matrices:

            X_train_neuron = (
                X_neuron[train_idx]
            )

            X_test_neuron = (
                X_neuron[test_idx]
            )

            y_train = (
                y[train_idx]
            )


            # ------------------------------------------------
            # Skip neuron if all temporal dimensions have
            # zero variance in the training fold.
            # ------------------------------------------------

            feature_variances = np.var(
                X_train_neuron,
                axis=0
            )

            if np.all(
                feature_variances == 0
            ):

                neurons_skipped += 1
                continue


            # ------------------------------------------------
            # Maximum possible LDA dimensions
            # ------------------------------------------------

            n_classes_train = len(
                np.unique(
                    y_train
                )
            )

            n_components = min(
                max_components,
                n_classes_train - 1,
                X_train_neuron.shape[1]
            )


            if n_components < 1:

                neurons_skipped += 1
                continue


            # ------------------------------------------------
            # SHRINKAGE LDA
            # ------------------------------------------------

            lda = LinearDiscriminantAnalysis(
                solver="eigen",
                shrinkage=gamma,
                n_components=n_components
            )


            try:

                lda.fit(
                    X_train_neuron,
                    y_train
                )


                train_components = (
                    lda.transform(
                        X_train_neuron
                    )
                )

                test_components = (
                    lda.transform(
                        X_test_neuron
                    )
                )


            except Exception:

                neurons_skipped += 1
                continue


            # ------------------------------------------------
            # SAFETY CHECK
            # ------------------------------------------------

            if (
                np.isnan(
                    train_components
                ).any()
                or
                np.isnan(
                    test_components
                ).any()
                or
                np.isinf(
                    train_components
                ).any()
                or
                np.isinf(
                    test_components
                ).any()
            ):

                neurons_skipped += 1
                continue


            train_component_blocks.append(
                train_components
            )

            test_component_blocks.append(
                test_components
            )

            neurons_used += 1


        # ====================================================
        # CONCATENATE COMPONENTS ACROSS NEURONS
        # ====================================================

        if len(
            train_component_blocks
        ) == 0:

            raise RuntimeError(
                f"Fold {fold_number}: "
                f"no usable neuron LDA components."
            )


        X_train_population = (
            np.hstack(
                train_component_blocks
            )
        )

        X_test_population = (
            np.hstack(
                test_component_blocks
            )
        )


        # ====================================================
        # POPULATION MULTINOMIAL LOGISTIC REGRESSION
        # ====================================================

        population_model = Pipeline(
            [
                (
                    "scaler",
                    StandardScaler()
                ),

                (
                    "classifier",
                    LogisticRegression(
                        C=1.0,
                        solver="lbfgs",
                        max_iter=5000
                    )
                )
            ]
        )


        population_model.fit(
            X_train_population,
            y[train_idx]
        )


        fold_pred = (
            population_model.predict(
                X_test_population
            )
        )


        y_pred[test_idx] = (
            fold_pred
        )


        fold_accuracy = np.mean(
            fold_pred
            ==
            y[test_idx]
        )


        fold_rows.append(
            {
                "fold":
                    fold_number,

                "n_train":
                    len(train_idx),

                "n_test":
                    len(test_idx),

                "n_neurons_used":
                    neurons_used,

                "n_neurons_skipped":
                    neurons_skipped,

                "n_population_features":
                    X_train_population.shape[1],

                "accuracy":
                    fold_accuracy
            }
        )


    # --------------------------------------------------------
    # FINAL CHECK
    # --------------------------------------------------------

    if np.any(
        y_pred == -1
    ):

        raise RuntimeError(
            "Some held-out samples were never predicted."
        )


    overall_accuracy = np.mean(
        y_pred
        ==
        y
    )


    return (
        overall_accuracy,
        y_pred,
        pd.DataFrame(
            fold_rows
        )
    )


print(
    "Corrected lda_component_population_cv() loaded."
)

Corrected lda_component_population_cv() loaded.


In [20]:
# ============================================================
# CELL 11: ALL-SUBJECT LDA-COMPONENT PERMUTATION TEST
#
# IMPORTANT:
#
# For every permutation:
#
#   1. Shuffle number labels
#   2. Create stratified CV folds from shuffled labels
#   3. Refit each neuron's temporal LDA using shuffled
#      TRAINING labels only
#   4. Project train + held-out data
#   5. Concatenate neuron LDA components
#   6. Fit population multinomial logistic regression
#   7. Predict held-out shuffled labels
#
# Therefore the entire supervised LDA pipeline is included
# inside the permutation null.
# ============================================================

N_LDA_PERMUTATIONS = 100

LDA_BIN_MS = 60
LDA_GAMMA = 0.5
LDA_MAX_COMPONENTS = 3


lda_permutation_rows = []

all_lda_null_accuracies = []


print("=" * 92)
print("ALL-SUBJECT LDA-COMPONENT PERMUTATION TEST")
print("=" * 92)

print(
    f"Permutations per subject: {N_LDA_PERMUTATIONS}"
)

print(
    f"Temporal bin:             {LDA_BIN_MS} ms"
)

print(
    f"LDA gamma:                {LDA_GAMMA}"
)

print(
    f"Max components/neuron:    {LDA_MAX_COMPONENTS}"
)


# ============================================================
# LOOP THROUGH SUBJECTS
# ============================================================

for subject_number, subject in enumerate(
    SUBJECTS,
    start=1
):

    print("\n" + "-" * 92)

    print(
        f"[{subject_number:2d}/11] "
        f"Processing {subject}"
    )

    print("-" * 92)


    # --------------------------------------------------------
    # LOAD SPIKES
    # --------------------------------------------------------

    spike_file = (
        find_arithmetic_spike_file(
            subject
        )
    )

    spikes = (
        load_arithmetic_spikes(
            spike_file
        )
    )


    # --------------------------------------------------------
    # GET MTL NEURONS
    # --------------------------------------------------------

    mtl_indices, subject_mtl_table = (
        get_subject_mtl_indices(
            subject,
            mtl_results
        )
    )


    # --------------------------------------------------------
    # PREPARE OPERAND PRESENTATIONS
    # --------------------------------------------------------

    behav, pooled = (
        prepare_arithmetic_subject(
            subject,
            spikes
        )
    )


    # --------------------------------------------------------
    # GET y
    #
    # We use make_population_features here simply to guarantee
    # exactly the same presentation ordering used previously.
    # --------------------------------------------------------

    X_fr_check, X_temp_check, y, metadata = (
        make_population_features(
            spikes=spikes,
            pooled_table=pooled,
            neuron_indices=mtl_indices,
            temporal_bin_ms=LDA_BIN_MS
        )
    )


    # ========================================================
    # OBSERVED LDA DECODING
    # ========================================================

    observed_splits = (
        make_stratified_cv_splits(
            y,
            random_state=RANDOM_STATE
        )
    )


    lda_obs, lda_pred_obs, lda_fold_obs = (
        lda_component_population_cv(
            spikes=spikes,
            pooled_table=pooled,
            neuron_indices=mtl_indices,
            y=y,
            splits=observed_splits,
            temporal_bin_ms=LDA_BIN_MS,
            gamma=LDA_GAMMA,
            max_components=LDA_MAX_COMPONENTS
        )
    )


    # ========================================================
    # PERMUTATION NULL
    # ========================================================

    null_lda = np.zeros(
        N_LDA_PERMUTATIONS
    )


    rng = np.random.default_rng(
        RANDOM_STATE
        +
        50000
        +
        subject_number * 1000
    )


    for perm in range(
        N_LDA_PERMUTATIONS
    ):

        # ----------------------------------------------------
        # SHUFFLE LABELS
        # ----------------------------------------------------

        y_perm = (
            rng.permutation(
                y
            )
        )


        # ----------------------------------------------------
        # STRATIFIED FOLDS FOR PERMUTED LABELS
        # ----------------------------------------------------

        perm_splits = (
            make_stratified_cv_splits(
                y_perm,
                random_state=RANDOM_STATE + perm
            )
        )


        # ----------------------------------------------------
        # CRITICAL:
        #
        # lda_component_population_cv() refits each neuron's
        # LDA inside every training fold using y_perm.
        #
        # Therefore true labels are NOT used here.
        # ----------------------------------------------------

        lda_perm, _, _ = (
            lda_component_population_cv(
                spikes=spikes,
                pooled_table=pooled,
                neuron_indices=mtl_indices,
                y=y_perm,
                splits=perm_splits,
                temporal_bin_ms=LDA_BIN_MS,
                gamma=LDA_GAMMA,
                max_components=LDA_MAX_COMPONENTS
            )
        )


        null_lda[perm] = (
            lda_perm
        )


        if (
            (perm + 1) % 20 == 0
        ):

            print(
                f"    permutations: "
                f"{perm + 1:3d}/"
                f"{N_LDA_PERMUTATIONS}"
            )


    # ========================================================
    # PERMUTATION P VALUE
    # ========================================================

    lda_p = (
        1
        +
        np.sum(
            null_lda
            >=
            lda_obs
        )
    ) / (
        N_LDA_PERMUTATIONS
        +
        1
    )


    # --------------------------------------------------------
    # STORE NULL
    # --------------------------------------------------------

    all_lda_null_accuracies.append(
        null_lda.copy()
    )


    # --------------------------------------------------------
    # STORE SUBJECT RESULT
    # --------------------------------------------------------

    lda_permutation_rows.append(
        {
            "subject":
                subject,

            "n_mtl_neurons":
                len(mtl_indices),

            "n_presentations":
                len(y),

            "lda_accuracy":
                lda_obs,

            "lda_null_mean":
                null_lda.mean(),

            "lda_null_sd":
                null_lda.std(
                    ddof=1
                ),

            "lda_perm_p":
                lda_p,

            "mean_neurons_used":
                lda_fold_obs[
                    "n_neurons_used"
                ].mean(),

            "mean_neurons_skipped":
                lda_fold_obs[
                    "n_neurons_skipped"
                ].mean(),

            "mean_population_features":
                lda_fold_obs[
                    "n_population_features"
                ].mean()
        }
    )


    # --------------------------------------------------------
    # PRINT SUBJECT RESULT
    # --------------------------------------------------------

    print()

    print(
        f"    Observed LDA:  "
        f"{100 * lda_obs:6.2f}%"
    )

    print(
        f"    Null mean:     "
        f"{100 * null_lda.mean():6.2f}%"
    )

    print(
        f"    Permutation p: "
        f"{lda_p:.4f}"
    )


# ============================================================
# CREATE SUMMARY TABLE
# ============================================================

lda_perm_results = (
    pd.DataFrame(
        lda_permutation_rows
    )
)


# ============================================================
# SAVE SUBJECT RESULTS
# ============================================================

lda_perm_file = (
    TABLES /
    "arithmetic_population_lda_components_60ms_permutation.csv"
)

lda_perm_results.to_csv(
    lda_perm_file,
    index=False
)


# ============================================================
# GROUP-LEVEL LDA ACCURACY
# ============================================================

observed_mean_lda = (
    lda_perm_results[
        "lda_accuracy"
    ].mean()
)


# Each row = subject
# Each column = permutation
lda_null_matrix = np.vstack(
    all_lda_null_accuracies
)


# Average independently shuffled decoding accuracies
# across the 11 subjects.
group_lda_null = (
    lda_null_matrix.mean(
        axis=0
    )
)


group_lda_p = (
    1
    +
    np.sum(
        group_lda_null
        >=
        observed_mean_lda
    )
) / (
    N_LDA_PERMUTATIONS
    +
    1
)


# ============================================================
# SAVE GROUP NULL
# ============================================================

lda_group_null_file = (
    TABLES /
    "arithmetic_population_lda_components_group_null.csv"
)

pd.DataFrame(
    {
        "permutation":
            np.arange(
                1,
                N_LDA_PERMUTATIONS + 1
            ),

        "mean_null_lda_accuracy":
            group_lda_null
    }
).to_csv(
    lda_group_null_file,
    index=False
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("SUBJECT-WISE LDA PERMUTATION SUMMARY")
print("=" * 100)


for _, row in lda_perm_results.iterrows():

    print(
        f"{row['subject']}:  "
        f"LDA={100 * row['lda_accuracy']:6.2f}%   "
        f"null={100 * row['lda_null_mean']:6.2f}%   "
        f"p={row['lda_perm_p']:.4f}"
    )


print("\n" + "=" * 100)
print("GROUP-LEVEL LDA DECODING")
print("=" * 100)

print(
    f"Mean observed LDA accuracy: "
    f"{100 * observed_mean_lda:.2f}%"
)

print(
    f"Mean permutation null:      "
    f"{100 * group_lda_null.mean():.2f}%"
)

print(
    f"Group permutation p-value:  "
    f"{group_lda_p:.4f}"
)

print()

print(
    "Subjects with significant LDA decoding "
    "(p < 0.05):",
    int(
        (
            lda_perm_results[
                "lda_perm_p"
            ]
            < 0.05
        ).sum()
    ),
    "/ 11"
)


print("\nSaved:")
print(lda_perm_file)
print(lda_group_null_file)

ALL-SUBJECT LDA-COMPONENT PERMUTATION TEST
Permutations per subject: 100
Temporal bin:             60 ms
LDA gamma:                0.5
Max components/neuron:    3

--------------------------------------------------------------------------------------------
[ 1/11] Processing YFF
--------------------------------------------------------------------------------------------
    permutations:  20/100
    permutations:  40/100
    permutations:  60/100
    permutations:  80/100
    permutations: 100/100

    Observed LDA:   20.18%
    Null mean:      13.15%
    Permutation p: 0.0297

--------------------------------------------------------------------------------------------
[ 2/11] Processing YFI
--------------------------------------------------------------------------------------------
    permutations:  20/100
    permutations:  40/100
    permutations:  60/100
    permutations:  80/100
    permutations: 100/100

    Observed LDA:   18.27%
    Null mean:      14.92%
    Permutation p: 0.

In [23]:
# ============================================================
# PRINT SAVED LDA PERMUTATION RESULTS
# NO COMPUTATION / NO PERMUTATIONS ARE RERUN
# ============================================================

print("=" * 100)
print("SUBJECT-WISE LDA PERMUTATION SUMMARY")
print("=" * 100)

for _, row in lda_perm_results.iterrows():

    print(
        f"{row['subject']}:  "
        f"LDA={100 * row['lda_accuracy']:6.2f}%   "
        f"null={100 * row['lda_null_mean']:6.2f}%   "
        f"p={row['lda_perm_p']:.4f}"
    )


print("\n" + "=" * 100)
print("GROUP-LEVEL LDA DECODING")
print("=" * 100)

print(
    f"Mean observed LDA accuracy: "
    f"{100 * observed_mean_lda:.2f}%"
)

print(
    f"Mean permutation null:      "
    f"{100 * group_lda_null.mean():.2f}%"
)

print(
    f"Group permutation p-value:  "
    f"{group_lda_p:.4f}"
)

print(
    f"Significant subjects:       "
    f"{(lda_perm_results['lda_perm_p'] < 0.05).sum()} / 11"
)


print("\n" + "=" * 100)
print("FINAL THREE-REPRESENTATION COMPARISON")
print("=" * 100)

print(
    f"FR population:               "
    f"{100 * population_perm_results['fr_accuracy'].mean():.2f}%"
)

print(
    f"Raw temporal (60 ms):        "
    f"{100 * population_perm_results['temporal_accuracy'].mean():.2f}%"
)

print(
    f"LDA temporal components:     "
    f"{100 * observed_mean_lda:.2f}%"
)

print(
    f"Nominal 9-class chance:      "
    f"{100/9:.2f}%"
)

SUBJECT-WISE LDA PERMUTATION SUMMARY
YFF:  LDA= 20.18%   null= 13.15%   p=0.0297
YFI:  LDA= 18.27%   null= 14.92%   p=0.2673
YFJ:  LDA= 10.53%   null= 13.93%   p=0.8614
YFK:  LDA= 21.05%   null= 13.10%   p=0.0396
YFL:  LDA=  6.14%   null= 13.33%   p=0.9802
YFM:  LDA= 13.16%   null= 13.75%   p=0.6139
YFP:  LDA= 11.39%   null= 13.32%   p=0.7426
YFR:  LDA= 14.66%   null= 11.68%   p=0.1386
YFS:  LDA=  9.47%   null= 11.36%   p=0.8218
YFT:  LDA= 11.74%   null= 11.51%   p=0.4653
YFU:  LDA= 10.73%   null= 11.27%   p=0.6337

GROUP-LEVEL LDA DECODING
Mean observed LDA accuracy: 13.39%
Mean permutation null:      12.85%
Group permutation p-value:  0.3267
Significant subjects:       2 / 11

FINAL THREE-REPRESENTATION COMPARISON
FR population:               12.59%
Raw temporal (60 ms):        14.87%
LDA temporal components:     13.39%
Nominal 9-class chance:      11.11%


In [22]:
1+1

2

In [24]:
# ============================================================
# FINAL ADVISOR-READY SUMMARY
# Single-neuron temporal advantage + population decoding
# ============================================================

print("=" * 105)
print("FINAL SUMMARY: FIRING RATE vs TEMPORAL POPULATION REPRESENTATIONS")
print("=" * 105)

chance = 1 / 9

# ------------------------------------------------------------
# Known single-neuron reproduction results
# ------------------------------------------------------------

single_fr = 0.10642
single_temporal = 0.14484
single_delta = single_temporal - single_fr
single_p = 6.1999e-9

# ------------------------------------------------------------
# Population results already computed
# ------------------------------------------------------------

pop_fr = population_perm_results["fr_accuracy"].mean()
pop_temporal = population_perm_results["temporal_accuracy"].mean()
pop_delta = pop_temporal - pop_fr

pop_delta_null = group_null_delta.mean()
pop_delta_p = group_delta_p

lda_obs = observed_mean_lda
lda_null = group_lda_null.mean()
lda_group_p = group_lda_p

n_lda_sig = int(
    (lda_perm_results["lda_perm_p"] < 0.05).sum()
)

n_fr_sig = int(
    (population_perm_results["fr_perm_p"] < 0.05).sum()
)

n_temp_sig = int(
    (population_perm_results["temporal_perm_p"] < 0.05).sum()
)


# ============================================================
# 1. BASIC QUESTION
# ============================================================

print("\n1. WHAT WAS THE QUESTION?")
print("-" * 105)

print("""
I wanted to compare different ways of representing the neural response to numbers.

Firing-rate representation:
    Collapse the entire 900-ms response window into one spike-count /
    firing-rate value per neuron.

Common-bin temporal representation:
    Preserve temporal structure by dividing the same 900-ms window into
    60-ms bins. This gives 15 temporal values per neuron.

LDA temporal representation:
    Start from the 60-ms temporal pattern for each neuron, fit an LDA
    using only the training data, keep up to 3 discriminant components
    per neuron, concatenate those components across simultaneously
    recorded neurons, and decode the number from that population vector.
""")


# ============================================================
# 2. SINGLE-NEURON RESULT
# ============================================================

print("\n2. SINGLE-NEURON RESULT")
print("-" * 105)

print(
    f"Average firing-rate decoding accuracy:   {100 * single_fr:.2f}%"
)

print(
    f"Average temporal decoding accuracy:      {100 * single_temporal:.2f}%"
)

print(
    f"Temporal improvement:                    "
    f"{100 * single_delta:+.2f} percentage points"
)

print(
    f"Paired subject-level p-value:            {single_p:.3e}"
)

print(f"Nominal 9-class chance:                   {100 * chance:.2f}%")

print("""
Interpretation:
At the single-neuron level, preserving the temporal pattern of spikes
clearly improved number decoding compared with collapsing the response
into a single firing-rate value.

The important result here is NOT simply that temporal accuracy was
14.48%. The stronger result is that temporal decoding was consistently
better than firing-rate decoding across subjects.

Therefore, for the single-neuron temporal-vs-firing-rate comparison:

    H0: Temporal representation provides no systematic decoding
        advantage over firing rate.

Because p << 0.05:

    DECISION: Reject H0.

Conclusion:
There is strong evidence for a single-neuron temporal advantage.
Temporal spike structure contains number-related information that is
lost when the whole response is reduced to one firing-rate value.
""")


# ============================================================
# 3. POPULATION RESULTS
# ============================================================

print("\n3. SUBJECT-WISE POPULATION DECODING")
print("-" * 105)

print(f"{'Representation':<32} {'Observed':>12} {'Null / comparison':>20} {'Result':>25}")
print("-" * 105)

print(
    f"{'Firing rate':<32}"
    f"{100 * pop_fr:>11.2f}% "
    f"{'permutation tested':>20} "
    f"{str(n_fr_sig) + '/11 sig. subjects':>25}"
)

print(
    f"{'Common 60-ms temporal':<32}"
    f"{100 * pop_temporal:>11.2f}% "
    f"{'permutation tested':>20} "
    f"{str(n_temp_sig) + '/11 sig. subjects':>25}"
)

print(
    f"{'Temporal - FR advantage':<32}"
    f"{100 * pop_delta:>+11.2f} pp "
    f"{100 * pop_delta_null:>+17.2f} pp "
    f"{('p = ' + format(pop_delta_p, '.4f')):>25}"
)

print(
    f"{'LDA temporal components':<32}"
    f"{100 * lda_obs:>11.2f}% "
    f"{100 * lda_null:>19.2f}% "
    f"{('p = ' + format(lda_group_p, '.4f')):>25}"
)


# ============================================================
# 4. POPULATION NULL HYPOTHESIS
# ============================================================

print("\n4. WHAT DOES THE POPULATION PERMUTATION TEST ASK?")
print("-" * 105)

print("""
For population decoding, the permutation null hypothesis is:

    H0:
    Neural population activity has no reliable decodable relationship
    with the number labels.

To simulate H0, I shuffled the number labels and reran the decoding
pipeline. Repeating this generates the null distribution.

The null mean is the average decoding performance obtained when the
number-neural relationship has been destroyed.

The permutation p-value tells us how often an accuracy at least as
large as the observed result occurs under this null distribution.
""")


# ============================================================
# 5. TEMPORAL vs FR POPULATION TEST
# ============================================================

print("\n5. DID TEMPORAL BINNING IMPROVE POPULATION DECODING?")
print("-" * 105)

print(
    f"Mean population FR accuracy:              {100 * pop_fr:.2f}%"
)

print(
    f"Mean population temporal accuracy:        {100 * pop_temporal:.2f}%"
)

print(
    f"Observed temporal - FR improvement:        "
    f"{100 * pop_delta:+.2f} pp"
)

print(
    f"Permutation-null mean improvement:         "
    f"{100 * pop_delta_null:+.2f} pp"
)

print(
    f"Permutation p-value:                       "
    f"{pop_delta_p:.4f}"
)

if pop_delta_p < 0.05:
    print("\nDECISION: Reject H0.")
else:
    print("\nDECISION: Fail to reject H0.")

print("""
Interpretation:
The temporal population representation was numerically better than
firing rate, but the improvement was not sufficiently unusual relative
to what occurred after label shuffling.

Therefore I cannot claim a robust temporal population advantage from
this analysis.
""")


# ============================================================
# 6. LDA POPULATION TEST
# ============================================================

print("\n6. DID THE LDA TEMPORAL POPULATION REPRESENTATION DECODE NUMBER?")
print("-" * 105)

print(
    f"Observed mean LDA accuracy:                {100 * lda_obs:.2f}%"
)

print(
    f"Permutation-null mean LDA accuracy:        {100 * lda_null:.2f}%"
)

print(
    f"Nominal chance:                            {100 * chance:.2f}%"
)

print(
    f"Group permutation p-value:                 {lda_group_p:.4f}"
)

print(
    f"Nominally significant individual subjects: {n_lda_sig}/11"
)

if lda_group_p < 0.05:
    print("\nDECISION: Reject H0 at the group level.")
else:
    print("\nDECISION: Fail to reject H0 at the group level.")

print("""
Interpretation:
Although the observed LDA accuracy is above theoretical 11.11% chance,
the permutation null itself is also elevated.

The observed mean was only slightly above the empirical null mean.
Therefore the group-level result is not statistically significant.

Two subjects (YFF and YFK) were nominally significant individually,
but the overall group result was not significant and those subject-wise
tests have not been treated as strong multiple-comparison-corrected
evidence.
""")


# ============================================================
# 7. WHY SINGLE NEURON REJECTS H0 BUT POPULATION DOES NOT
# ============================================================

print("\n7. WHY IS THE SINGLE-NEURON RESULT STRONGER THAN THE POPULATION RESULT?")
print("-" * 105)

print("""
This does NOT mean that a single neuron necessarily decodes better than
the whole population.

The two analyses test different statistical questions.

SINGLE-NEURON RESULT
--------------------
The strongest single-neuron result is a paired comparison:

    Temporal representation  vs  firing-rate representation

for the same neurons.

The temporal improvement was highly consistent across subjects.
That consistency produced extremely strong evidence against the null
hypothesis of no temporal advantage.

POPULATION RESULT
-----------------
The population question is:

    Can simultaneously recorded population activity reliably predict
    the number on held-out presentations?

This is harder statistically.

The raw temporal population representation can contain hundreds of
features. For example, 50 neurons x 15 temporal bins gives 750 input
features, while a subject may have only roughly hundreds of operand
presentations.

Adding neurons therefore adds both:

    useful signal + additional dimensions/noise.

More neurons do not automatically mean better generalization.

The permutation test showed that decoding accuracies around the observed
values can also occur reasonably often after destroying the true
number-label relationship. Therefore the observed population accuracy
was not sufficiently separated from its empirical null distribution.
""")


# ============================================================
# 8. FINAL CONCLUSION
# ============================================================

print("\n8. FINAL TAKEAWAY")
print("=" * 105)

print(f"""
SINGLE-NEURON LEVEL
-------------------
FR decoding:             {100 * single_fr:.2f}%
Temporal decoding:       {100 * single_temporal:.2f}%
Temporal advantage:      {100 * single_delta:+.2f} pp
p-value:                 {single_p:.3e}

=> Reject the null hypothesis of no temporal advantage.

Strong evidence:
Temporal spike patterns provide more number-related information than
a fixed-window firing-rate representation at the single-neuron level.


POPULATION LEVEL
----------------
FR population decoding:              {100 * pop_fr:.2f}%
Common-bin temporal decoding:        {100 * pop_temporal:.2f}%
LDA temporal-component decoding:     {100 * lda_obs:.2f}%

Temporal - FR permutation p:         {pop_delta_p:.4f}
LDA group permutation p:             {lda_group_p:.4f}

=> Fail to reject the relevant population-level null hypotheses.

Current conclusion:
I found a clear and robust temporal advantage at the single-neuron level,
but I did not find robust group-level evidence for population decoding
with the population representations tested so far.

This should NOT be stated as:
    "There is no population coding."

The correct statement is:
    "With these subject-wise population decoding implementations,
     I have not yet established statistically significant group-level
     population decoding."

Also, this exploratory population LDA decoder is not identical to the
paper's simplex representation. The simplex analysis uses selected
number-responsive neurons and their individually optimized temporal
representations before constructing population geometry.
""")

print("=" * 105)

FINAL SUMMARY: FIRING RATE vs TEMPORAL POPULATION REPRESENTATIONS

1. WHAT WAS THE QUESTION?
---------------------------------------------------------------------------------------------------------

I wanted to compare different ways of representing the neural response to numbers.

Firing-rate representation:
    Collapse the entire 900-ms response window into one spike-count /
    firing-rate value per neuron.

Common-bin temporal representation:
    Preserve temporal structure by dividing the same 900-ms window into
    60-ms bins. This gives 15 temporal values per neuron.

LDA temporal representation:
    Start from the 60-ms temporal pattern for each neuron, fit an LDA
    using only the training data, keep up to 3 discriminant components
    per neuron, concatenate those components across simultaneously
    recorded neurons, and decode the number from that population vector.


2. SINGLE-NEURON RESULT
--------------------------------------------------------------------------------

In [ ]:
# ============================================================
# FINAL NUMERICAL RESULTS
# ============================================================

print("=" * 72)
print("SINGLE-NEURON DECODING")
print("=" * 72)

print("9-class chance level:          11.11%")
print("Mean firing-rate accuracy:     10.64%")
print("Mean temporal accuracy:        14.48%")
print("Temporal advantage:            +3.84 percentage points")
print("Paired subject-level p-value:  6.20e-09")

print("""
H0: Temporal decoding does not systematically outperform
    firing-rate decoding at the single-neuron level.

p << 0.05  ->  REJECT H0

Interpretation:
There is strong evidence that preserving the temporal spike pattern
improves single-neuron number decoding compared with using only the
total firing rate.
""")


print("=" * 72)
print("POPULATION DECODING")
print("=" * 72)

print("9-class chance level:          11.11%")
print()
print("Mean firing-rate accuracy:     12.59%")
print("Mean 60-ms temporal accuracy:  14.87%")
print("Mean LDA temporal accuracy:    13.39%")

print()
print("Temporal - FR advantage:       +2.28 percentage points")
print("Permutation-null advantage:    +0.86 percentage points")
print("Temporal - FR perm. p-value:    0.1188")

print()
print("LDA observed accuracy:         13.39%")
print("LDA permutation-null mean:     12.85%")
print("LDA group perm. p-value:       0.3267")

print()
print("Significant subjects:")
print("  FR:                          0 / 11")
print("  60-ms temporal:              0 / 11")
print("  LDA temporal:                2 / 11 (YFF, YFK; nominal)")


print("""
Population H0:
Neural population activity has no reliable decodable relationship
with number labels.

Temporal vs FR:
    p = 0.1188 > 0.05  ->  FAIL TO REJECT H0

LDA population:
    p = 0.3267 > 0.05  ->  FAIL TO REJECT H0

Interpretation:
Population temporal accuracy was numerically highest (14.87%), but
the permutation tests did not provide robust group-level evidence
for population decoding.

Important:
This does NOT prove that population number information is absent.
It means that these population decoding analyses did not provide
sufficient evidence to reject the null hypothesis.
""")


print("=" * 72)
print("BOTTOM LINE")
print("=" * 72)

print("""
Single neuron:
    Strong temporal advantage -> H0 rejected.

Population:
    FR       = 12.59%
    Temporal = 14.87%
    LDA      = 13.39%

    No robust group-level significance -> H0 not rejected.

Therefore:
Strong evidence for temporal information at the single-neuron level,
but not robust evidence for population decoding with the current
population implementations.
""")

SINGLE-NEURON DECODING
9-class chance level:          11.11%
Mean firing-rate accuracy:     10.64%
Mean temporal accuracy:        14.48%
Temporal advantage:            +3.84 percentage points
Paired subject-level p-value:  6.20e-09

H0: Temporal decoding does not systematically outperform
    firing-rate decoding at the single-neuron level.

p << 0.05  ->  REJECT H0

Interpretation:
There is strong evidence that preserving the temporal spike pattern
improves single-neuron number decoding compared with using only the
total firing rate.

POPULATION DECODING
9-class chance level:          11.11%

Mean firing-rate accuracy:     12.59%
Mean 60-ms temporal accuracy:  14.87%
Mean LDA temporal accuracy:    13.39%

Temporal - FR advantage:       +2.28 percentage points
Permutation-null advantage:    +0.86 percentage points
Temporal - FR perm. p-value:    0.1188

LDA observed accuracy:         13.39%
LDA permutation-null mean:     12.85%
LDA group perm. p-value:       0.3267

Significant subject

: 